# 🏀 NCAAB Daily Game Predictor
**Predicts win probability, implied moneyline odds, and compares to Vegas spread for today's college basketball games.**

### Pipeline:
1. Fetch today's NCAAB schedule from ESPN
2. Pull 2024-25 & 2025-26 season game logs per team from ESPN
3. Fetch team rankings (AP Poll / NET)
4. Run sentiment analysis on recent news (Google News RSS + ESPN)
5. Engineer features per matchup
6. Train & compare 3 ML models: Linear Regression, SGD Regressor, Neural Net
7. Output formatted results table with win probability, moneyline, vs. Vegas spread

# Set up

In [1]:
from google.colab import drive

drive.mount('/content/drive', force_remount=False)

Mounted at /content/drive


## Cell 1 — Install Dependencies

In [2]:
!pip install -q requests beautifulsoup4 pandas numpy scikit-learn tensorflow nltk vaderSentiment feedparser lxml tabulate

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 126.0/126.0 kB 6.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 81.5/81.5 kB 4.4 MB/s eta 0:00:00


## Cell 2 — Imports & Setup

In [3]:
import requests
import feedparser
import pandas as pd
import numpy as np
import json
import time
import re
import warnings
warnings.filterwarnings('ignore')

from bs4 import BeautifulSoup
from datetime import datetime, date
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer

from sklearn.linear_model import LinearRegression, SGDRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.model_selection import cross_val_score
from sklearn.metrics import mean_absolute_error

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

from IPython.display import display, HTML
from tabulate import tabulate

import nltk
nltk.download('vader_lexicon', quiet=True)

HEADERS = {
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36'
}

TODAY = date.today().strftime('%Y%m%d')
print(f"✅ Setup complete. Today's date: {TODAY}")

✅ Setup complete. Today's date: 20260302


## Cell 3 — Fetch Today's NCAAB Schedule from ESPN

In [4]:
# =============================================================================
# CELL 3 — Fetch Today's NCAAB Schedule from CBS Sports
# =============================================================================

import requests
import re
import pandas as pd
from bs4 import BeautifulSoup
from datetime import date

TODAY   = date.today().strftime('%Y%m%d')
HEADERS = {
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) '
                  'AppleWebKit/537.36 (KHTML, like Gecko) '
                  'Chrome/120.0.0.0 Safari/537.36',
    'Accept':          'text/html,application/xhtml+xml,application/xml;q=0.9,*/*;q=0.8',
    'Accept-Language': 'en-US,en;q=0.9',
}


def get_todays_games():
    """
    Scrapes today's NCAAB schedule from CBS Sports scoreboard.
    URL: https://www.cbssports.com/college-basketball/scoreboard/FBS/{TODAY}/

    Parsing strategy (same as training data Cell 2):
      - Each game lives in <div class="single-score-card">
      - data-abbrev="NCAAB_YYYYMMDD_AWAY@HOME" gives away/home abbrevs
      - Row order in tbody is always: row 0 = away, row 1 = home
        (winner/loser classes are injected by JS and not available
         in static HTML — all rows show class='tiedGame' pre-render)
      - Team name from <a class="team-name-link"> within each row
      - Score from <td class="total"> within each row
      - Game status from <div class="game-status">
    """
    url   = f"https://www.cbssports.com/college-basketball/scoreboard/FBS/{TODAY}/"
    games = []

    print(f"🔍 Fetching NCAAB schedule for {TODAY}...")
    print(f"   URL: {url}\n")

    try:
        r = requests.get(url, headers=HEADERS, timeout=20)
        r.raise_for_status()
        soup = BeautifulSoup(r.text, 'lxml')

        game_cards = soup.find_all('div', class_='single-score-card')
        print(f"  CBS: {len(game_cards)} game cards found")

        for card in game_cards:
            try:
                # ── 1. away/home abbrevs from data-abbrev ─────────────────────
                abbrev   = card.get('data-abbrev', '')
                at_chunk = abbrev.split('_')[-1]
                if '@' not in at_chunk:
                    continue
                away_abbr, home_abbr = at_chunk.split('@', 1)

                # ── 2. game status ────────────────────────────────────────────
                status_div = card.find('div', class_='game-status')
                status_raw = status_div.get_text(strip=True) \
                             if status_div else 'Scheduled'
                is_final   = 'final' in status_raw.lower()

                # ── 3. score table rows ───────────────────────────────────────
                # Skip header row (class=None), keep data rows (class=tiedGame)
                all_trs = [tr for tr in card.find_all('tr')
                           if tr.get('class') is not None]

                if len(all_trs) < 2:
                    continue

                away_row = all_trs[0]
                home_row = all_trs[1]

                # ── 4. extract name and score ─────────────────────────────────
                def get_name(row):
                    a = row.find('a', class_='team-name-link')
                    return a.get_text(strip=True) if a else None

                def get_score(row):
                    td = row.find('td', class_='total')
                    if td:
                        val = td.get_text(strip=True)
                        return int(val) if val.isdigit() else None
                    return None

                away_name  = get_name(away_row)
                away_score = get_score(away_row)
                home_name  = get_name(home_row)
                home_score = get_score(home_row)

                if not away_name or not home_name:
                    continue

                # ── 5. winner/loser from score comparison ─────────────────────
                winner_name  = None
                winner_score = None
                loser_name   = None
                loser_score  = None
                home_team_won = None

                if is_final and away_score is not None and home_score is not None:
                    if home_score > away_score:
                        winner_name,  winner_score = home_name,  home_score
                        loser_name,   loser_score  = away_name,  away_score
                        home_team_won = 1
                    elif away_score > home_score:
                        winner_name,  winner_score = away_name,  away_score
                        loser_name,   loser_score  = home_name,  home_score
                        home_team_won = 0

                # ── 6. vegas spread from bottom bar ───────────────────────────
                vegas_spread = None
                bottom_bar   = card.find('div', class_='bottom-bar')
                if bottom_bar:
                    spread_match = re.search(
                        r'([+-]?\d+\.?\d*)', bottom_bar.get_text()
                    )
                    if spread_match:
                        vegas_spread = spread_match.group(1)

                games.append({
                    'game_id':      abbrev,
                    'away_name':    away_name,
                    'away_abbr':    away_abbr,
                    'away_score':   away_score,
                    'home_name':    home_name,
                    'home_abbr':    home_abbr,
                    'home_score':   home_score,
                    'status':       status_raw,
                    'home_team_won':home_team_won,
                    'winner_name':  winner_name,
                    'winner_score': winner_score,
                    'loser_name':   loser_name,
                    'loser_score':  loser_score,
                    'vegas_spread': vegas_spread,
                })

            except Exception as e:
                print(f"    ⚠️  Card parse error ({abbrev}): {e}")
                continue

    except Exception as e:
        print(f"  ❌ Failed to fetch schedule: {e}")

    # ── Summary ───────────────────────────────────────────────────────────────
    final_count     = sum(1 for g in games if g['home_team_won'] is not None)
    scheduled_count = sum(1 for g in games if g['home_team_won'] is None)

    print(f"\n{'='*55}")
    print(f"  Total games  : {len(games)}")
    print(f"  Final        : {final_count}")
    print(f"  Scheduled    : {scheduled_count}")
    print(f"{'='*55}")

    return games


# ── RUN ───────────────────────────────────────────────────────────────────────
games_today = get_todays_games()

if games_today:
    preview_df   = pd.DataFrame(games_today)
    display_cols = [
        'game_id', 'away_name', 'away_score',
        'home_name', 'home_score', 'status',
        'winner_name', 'winner_score',
        'loser_name',  'loser_score',
        'vegas_spread'
    ]
    display(preview_df[display_cols])
else:
    print("No games found today.")

🔍 Fetching NCAAB schedule for 20260302...
   URL: https://www.cbssports.com/college-basketball/scoreboard/FBS/20260302/

  CBS: 18 game cards found

  Total games  : 18
  Final        : 0
  Scheduled    : 18


,game_id,away_name,away_score,home_name,home_score,status,winner_name,winner_score,loser_name,loser_score,vegas_spread
0,NCAAB_20260302_NORFLK@MORGAN,Norfolk St.,None,Morgan St.,None,,None,None,None,None,None
1,NCAAB_20260302_COPPST@HOW,Coppin St.,None,Howard,None,,None,None,None,None,None
2,NCAAB_20260302_DUKE@NCST,Duke,None,NC State,None,,None,None,None,None,None
3,NCAAB_20260302_IUI@CLEVST,IUI,None,Clev. St.,None,,None,None,None,None,None
4,NCAAB_20260302_NCCU@UMES,NC Central,None,Md.-E. Shore,None,,None,None,None,None,None
5,NCAAB_20260302_SCST@DELST,SC State,None,Delaware St.,None,,None,None,None,None,None
6,NCAAB_20260302_MCNSE@NICHST,McNeese,None,Nicholls,None,,None,None,None,None,None
7,NCAAB_20260302_NWST@TEXPA,Northwestern St.,None,UT-Rio Grande Valley,None,,None,None,None,None,None
8,NCAAB_20260302_SFA@UIW,SF Austin,None,Incarnate Word,None,,None,None,None,None,None
9,NCAAB_20260302_MNTNA@NCOLO,Montana,None,N. Colorado,None,,None,None,None,None,None


# Pull Online Data

## Cell 4

In [5]:
# =============================================================================
# CELL 4 — Sentiment & News Fetcher: Google News RSS + ESPN Search
# Men's basketball only — women's basketball explicitly excluded
# =============================================================================

import feedparser
import requests
import re
import time
import numpy as np
from bs4 import BeautifulSoup
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer

analyzer = SentimentIntensityAnalyzer()

HEADERS = {
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) '
                  'AppleWebKit/537.36 (KHTML, like Gecko) '
                  'Chrome/120.0.0.0 Safari/537.36'
}

# ── Women's basketball filter — any article matching these is dropped ──────────
WOMENS_FILTER_TERMS = [
    "women's basketball", "womens basketball", "women's ncaa", "womens ncaa",
    "wnba", "w-nba", "lady ", "ladies ", "wbb ", " wbb",
    "ncaa women", "women's college basketball", "womens college basketball",
    "girls basketball", "female basketball", "women's team", "womens team",
    "she ", " her ", " hers ", "women's march madness",
]

def is_womens_article(title, full_text=""):
    """
    Returns True if the article appears to be about women's basketball.
    Checks both title and a sample of the body text.
    Title match alone is sufficient to reject — body match requires 2+ hits
    to avoid false positives (e.g. 'her' appearing incidentally in a men's article).
    """
    title_lower = title.lower()
    body_lower  = (full_text[:2000] if full_text else "").lower()  # only check first 2000 chars

    # Reject on title match alone
    for term in WOMENS_FILTER_TERMS:
        if term in title_lower:
            return True

    # Reject on body only if 2+ distinct terms match (avoids false positives)
    body_hits = sum(1 for term in WOMENS_FILTER_TERMS if term in body_lower)
    return body_hits >= 2


# ── Keyword dictionaries for tagging article text ─────────────────────────────
INJURY_KEYWORDS     = ['injur', 'hurt', 'out for', 'day-to-day', 'doubtful',
                       'questionable', 'sidelined', 'knee', 'ankle', 'concussion',
                       'surgery', 'absence', 'missed', 'will not play', 'won\'t play']

LINEUP_KEYWORDS     = ['starting lineup', 'starting five', 'lineup change',
                       'inserted into the starting', 'moved to the bench',
                       'benched', 'starter', 'rotation change', 'depth chart',
                       'replacing', 'new starter']

WIN_KEYWORDS        = ['win', 'victory', 'beat', 'defeated', 'upset', 'dominant',
                       'blowout', 'rolled', 'cruised', 'overcame', 'rallied',
                       'comeback', 'unbeaten', 'streak']

LOSS_KEYWORDS       = ['loss', 'lost', 'defeated', 'fell to', 'blown out',
                       'collapse', 'losing streak', 'skid', 'slide',
                       'struggle', 'winless']

MOMENTUM_KEYWORDS   = ['hot', 'on fire', 'rolling', 'clicking', 'momentum',
                       'confidence', 'energized', 'surging', 'impressive run',
                       'on a roll', 'winning streak']

SLUMP_KEYWORDS      = ['slump', 'cold', 'struggling', 'disappointing', 'slow',
                       'inconsistent', 'frustrating', 'skidding', 'dropped',
                       'winless', 'rough patch']

COACHING_KEYWORDS   = ['coach', 'head coach', 'staff', 'scheme', 'strategy',
                       'adjustment', 'system', 'game plan', 'coaching staff',
                       'fired', 'hired', 'contract', 'press conference']

RANKING_KEYWORDS    = ['ranked', 'ranking', 'top 25', 'ap poll', 'net ranking',
                       'bracketology', 'seed', 'projection', 'poll', 'ballot',
                       'moved up', 'dropped', 'climbed']

FATIGUE_KEYWORDS    = ['back-to-back', 'travel', 'tired', 'rest', 'fatigue',
                       'load management', 'minutes', 'heavy schedule',
                       'third game', 'quick turnaround']

HOME_AWAY_KEYWORDS  = ['home court', 'home crowd', 'road game', 'away game',
                       'hostile environment', 'sold out', 'neutral site',
                       'home advantage', 'visiting']


def keyword_score(text, keywords):
    """
    Returns a normalized score (0–1) for how strongly a text
    matches a keyword list. Higher = more mentions.
    """
    text_lower = text.lower()
    hits = sum(1 for kw in keywords if kw in text_lower)
    return min(hits / max(len(keywords) * 0.15, 1), 1.0)


def fetch_google_news_articles(team_name, max_articles=20, top_links=5):
    """
    Fetches up to max_articles from Google News RSS for a team.
    Query is scoped to men's college basketball and explicitly excludes
    women's basketball at the query level. A post-fetch filter then drops
    any remaining women's articles that slipped through.

    Returns list of dicts: {title, full_text, source_url, source}
    """
    # ── Men's-only query with explicit women's exclusion ──────────────────────
    # "mens" and NCAAB push toward men's results
    # -womens -WNBA -"women's" at the RSS level further filters
    query = requests.utils.quote(
        f'"{team_name}" mens college basketball NCAAB -womens -WNBA -"women\'s"'
    )
    url      = f"https://news.google.com/rss/search?q={query}&hl=en-US&gl=US&ceid=US:en"
    articles = []
    filtered_count = 0

    try:
        feed    = feedparser.parse(url)
        entries = feed.entries[:max_articles + 10]   # fetch extra to account for filtered items

        fetch_count = 0
        for entry in entries:
            if len(articles) >= max_articles:
                break

            title      = entry.get('title', '')
            source_url = entry.get('link', '')
            full_text  = title   # baseline fallback

            # ── Post-fetch women's filter (title check before fetching body) ──
            if is_womens_article(title):
                filtered_count += 1
                continue

            # ── For the top N links, fetch the raw page ────────────────────
            if fetch_count < top_links and source_url:
                try:
                    r = requests.get(
                        source_url,
                        headers=HEADERS,
                        timeout=10,
                        allow_redirects=True
                    )
                    r.raise_for_status()
                    raw_body  = r.text

                    # ── Body-level women's filter ──────────────────────────
                    if is_womens_article(title, raw_body):
                        filtered_count += 1
                        continue

                    full_text = f"{title} {raw_body}"
                    print(f"      📄 Fetched raw HTML ({len(raw_body):,} chars): {title[:60]}...")
                    fetch_count += 1

                except requests.exceptions.TooManyRedirects:
                    print(f"      ⚠️  Redirect loop skipped: {source_url[:60]}...")
                except requests.exceptions.Timeout:
                    print(f"      ⚠️  Timed out fetching: {source_url[:60]}...")
                except Exception as e:
                    print(f"      ⚠️  Could not fetch article: {e}")

            articles.append({
                'title':      title,
                'full_text':  full_text,
                'source_url': source_url,
                'source':     'google_news_full' if fetch_count <= top_links else 'google_news'
            })

    except Exception as e:
        print(f"    ⚠️  Google News fetch failed for {team_name}: {e}")

    if filtered_count > 0:
        print(f"      🚫 Filtered {filtered_count} women's basketball article(s)")

    return articles


def fetch_espn_news_articles(team_name, max_articles=15):
    """
    Searches ESPN's news search endpoint by team name.
    Scoped to mens-college-basketball sport parameter — ESPN's API already
    filters by sport, but we apply the women's title filter as a safety net.
    Returns list of dicts: {title, summary, full_text, source_url}
    """
    query = requests.utils.quote(f"{team_name} men's college basketball")
    url   = f"https://site.api.espn.com/apis/common/v3/search?query={query}&limit={max_articles}&type=article&sport=mens-college-basketball"
    articles = []
    filtered_count = 0
    try:
        r    = requests.get(url, headers=HEADERS, timeout=12)
        data = r.json()
        for item in data.get('results', [])[:max_articles]:
            title       = item.get('headline', item.get('title', ''))
            description = item.get('description', item.get('summary', ''))
            full_text   = f"{title} {description}"

            # Safety net filter even though ESPN API is scoped to men's
            if is_womens_article(title, full_text):
                filtered_count += 1
                continue

            articles.append({
                'title':     title,
                'summary':   description,
                'full_text': full_text,
                'source':    'espn'
            })
    except Exception as e:
        print(f"    ⚠️  ESPN news fetch failed for {team_name}: {e}")

    if filtered_count > 0:
        print(f"      🚫 ESPN filtered {filtered_count} women's article(s)")
    return articles


def fetch_espn_injury_articles(team_name, max_articles=10):
    """
    Targeted ESPN search specifically for men's injury news.
    """
    query = requests.utils.quote(f"{team_name} men's basketball injury OR injured OR out")
    url   = f"https://site.api.espn.com/apis/common/v3/search?query={query}&limit={max_articles}&type=article&sport=mens-college-basketball"
    articles = []
    try:
        r    = requests.get(url, headers=HEADERS, timeout=12)
        data = r.json()
        for item in data.get('results', [])[:max_articles]:
            title       = item.get('headline', '')
            description = item.get('description', '')
            full_text   = f"{title} {description}"
            if is_womens_article(title, full_text):
                continue
            articles.append({
                'title':     title,
                'summary':   description,
                'full_text': full_text,
                'source':    'espn_injury'
            })
    except Exception:
        pass
    return articles


def fetch_espn_lineup_articles(team_name, max_articles=10):
    """
    Targeted ESPN search for men's lineup/roster changes.
    """
    query = requests.utils.quote(f"{team_name} men's basketball lineup OR starter OR rotation OR benched")
    url   = f"https://site.api.espn.com/apis/common/v3/search?query={query}&limit={max_articles}&type=article&sport=mens-college-basketball"
    articles = []
    try:
        r    = requests.get(url, headers=HEADERS, timeout=12)
        data = r.json()
        for item in data.get('results', [])[:max_articles]:
            title       = item.get('headline', '')
            description = item.get('description', '')
            full_text   = f"{title} {description}"
            if is_womens_article(title, full_text):
                continue
            articles.append({
                'title':     title,
                'summary':   description,
                'full_text': full_text,
                'source':    'espn_lineup'
            })
    except Exception:
        pass
    return articles


print("✅ Cell 4 loaded — men's basketball only news fetcher defined.")
print(f"   Women's filter: {len(WOMENS_FILTER_TERMS)} exclusion terms active")
print(f"   Filtering applied at: query level + title level + body level")
print(f"   Keyword categories: injury, lineup, win, loss, momentum, slump,")
print(f"   coaching, ranking, fatigue, home/away ({sum([len(INJURY_KEYWORDS), len(LINEUP_KEYWORDS), len(WIN_KEYWORDS), len(LOSS_KEYWORDS), len(MOMENTUM_KEYWORDS), len(SLUMP_KEYWORDS), len(COACHING_KEYWORDS), len(RANKING_KEYWORDS), len(FATIGUE_KEYWORDS), len(HOME_AWAY_KEYWORDS)])} total keywords)")

✅ Cell 4 loaded — men's basketball only news fetcher defined.
   Women's filter: 21 exclusion terms active
   Filtering applied at: query level + title level + body level
   Keyword categories: injury, lineup, win, loss, momentum, slump,
   coaching, ranking, fatigue, home/away (118 total keywords)


## Cell 5

In [6]:
# =============================================================================
# CELL 5 — Feature Extractor: Build 25+ Named Features Per Team
# =============================================================================

def extract_team_features(team_name):
    """
    Given only a team name string, fetches all available news and
    computes 25+ named features grouped into 6 categories:

    [A] Sentiment Scores      — overall, ESPN-specific, Google-specific,
                                 headline-only, body-only, recent (last 5 articles)
    [B] Injury Signals        — injury mention rate, severity score,
                                 key-player injury flag, injury article count
    [C] Lineup / Roster       — lineup change signal, starter mention rate,
                                 roster instability score
    [D] Momentum / Form       — win mention rate, loss mention rate,
                                 momentum score, slump score, net momentum
    [E] Context / Environment — coaching stability, ranking mention rate,
                                 fatigue signal, home/away context mentions
    [F] Volume / Confidence   — total articles found, source diversity,
                                 data confidence score
    """

    # ── Fetch all articles ────────────────────────────────────────────────────
    gn_articles      = fetch_google_news_articles(team_name, max_articles=20)
    espn_articles    = fetch_espn_news_articles(team_name, max_articles=15)
    injury_articles  = fetch_espn_injury_articles(team_name, max_articles=10)
    lineup_articles  = fetch_espn_lineup_articles(team_name, max_articles=10)

    all_articles     = gn_articles + espn_articles
    all_texts        = [a['full_text'] for a in all_articles]
    headline_texts   = [a['title']     for a in all_articles]
    gn_texts         = [a['full_text'] for a in gn_articles]
    espn_texts       = [a['full_text'] for a in espn_articles]
    injury_texts     = [a['full_text'] for a in injury_articles]
    lineup_texts     = [a['full_text'] for a in lineup_articles]

    total_articles   = len(all_articles)

    def mean_vader(texts):
        if not texts:
            return 0.0
        scores = [analyzer.polarity_scores(t)['compound'] for t in texts]
        return float(np.mean(scores))

    def count_keyword_hits(texts, keywords):
        """Returns fraction of texts that contain at least one keyword."""
        if not texts:
            return 0.0
        hits = sum(1 for t in texts
                   if any(kw in t.lower() for kw in keywords))
        return hits / len(texts)

    def max_keyword_hits(texts, keywords):
        """Returns the highest keyword density found in any single article."""
        if not texts:
            return 0.0
        return max((keyword_score(t, keywords) for t in texts), default=0.0)

    # ─────────────────────────────────────────────────────────────────────────
    # [A] SENTIMENT SCORES
    # ─────────────────────────────────────────────────────────────────────────
    feat_sentiment_overall      = mean_vader(all_texts)
    feat_sentiment_espn         = mean_vader(espn_texts)
    feat_sentiment_google       = mean_vader(gn_texts)
    feat_sentiment_headlines    = mean_vader(headline_texts)
    # Recency: weight last 5 articles more heavily
    recent_texts                = all_texts[-5:] if len(all_texts) >= 5 else all_texts
    feat_sentiment_recent       = mean_vader(recent_texts)
    # Positive / negative article ratio
    all_scores                  = [analyzer.polarity_scores(t)['compound']
                                   for t in all_texts] if all_texts else [0.0]
    feat_pct_positive_articles  = sum(1 for s in all_scores if s >  0.05) / max(len(all_scores), 1)
    feat_pct_negative_articles  = sum(1 for s in all_scores if s < -0.05) / max(len(all_scores), 1)

    # ─────────────────────────────────────────────────────────────────────────
    # [B] INJURY SIGNALS
    # ─────────────────────────────────────────────────────────────────────────
    combined_injury_texts        = all_texts + injury_texts
    feat_injury_mention_rate     = count_keyword_hits(combined_injury_texts, INJURY_KEYWORDS)
    feat_injury_severity_score   = max_keyword_hits(combined_injury_texts,   INJURY_KEYWORDS)
    feat_injury_article_count    = len(injury_articles) / 10.0   # normalized 0–1
    # Key-player injury flag: "star", "leading scorer", "best player" + injury keyword
    star_injury_pattern          = re.compile(
        r'(star|leading scorer|best player|top player|key player|starting).{0,40}'
        r'(injur|out|sidelined|miss)',
        re.IGNORECASE
    )
    feat_key_player_injury_flag  = float(any(
        star_injury_pattern.search(t) for t in combined_injury_texts
    ))
    # Injury sentiment: how negative is the injury-specific coverage?
    feat_injury_sentiment        = mean_vader(injury_texts) if injury_texts else 0.0

    # ─────────────────────────────────────────────────────────────────────────
    # [C] LINEUP / ROSTER SIGNALS
    # ─────────────────────────────────────────────────────────────────────────
    combined_lineup_texts        = all_texts + lineup_texts
    feat_lineup_change_signal    = count_keyword_hits(combined_lineup_texts, LINEUP_KEYWORDS)
    feat_starter_mention_rate    = count_keyword_hits(combined_lineup_texts,
                                                      ['starter', 'starting'])
    feat_benched_signal          = count_keyword_hits(combined_lineup_texts,
                                                      ['bench', 'benched', 'moved to bench'])
    # Roster instability: combo of lineup changes + injuries
    feat_roster_instability      = min(
        (feat_lineup_change_signal * 0.5 + feat_injury_mention_rate * 0.5), 1.0
    )
    feat_lineup_sentiment        = mean_vader(lineup_texts) if lineup_texts else 0.0

    # ─────────────────────────────────────────────────────────────────────────
    # [D] MOMENTUM / FORM SIGNALS
    # ─────────────────────────────────────────────────────────────────────────
    feat_win_mention_rate        = count_keyword_hits(all_texts, WIN_KEYWORDS)
    feat_loss_mention_rate       = count_keyword_hits(all_texts, LOSS_KEYWORDS)
    feat_momentum_score          = count_keyword_hits(all_texts, MOMENTUM_KEYWORDS)
    feat_slump_score             = count_keyword_hits(all_texts, SLUMP_KEYWORDS)
    # Net momentum: positive = good form, negative = bad form
    feat_net_momentum            = feat_momentum_score - feat_slump_score
    # Win/loss ratio in coverage
    feat_win_loss_coverage_ratio = feat_win_mention_rate / max(feat_loss_mention_rate, 0.01)

    # ─────────────────────────────────────────────────────────────────────────
    # [E] CONTEXT / ENVIRONMENT SIGNALS
    # ─────────────────────────────────────────────────────────────────────────
    feat_coaching_mention_rate   = count_keyword_hits(all_texts, COACHING_KEYWORDS)
    # Coaching instability: "fired", "resign", "leave" near "coach"
    coaching_instability_pattern = re.compile(
        r'(coach).{0,30}(fired|resign|left|replaced|interim|stepping down)',
        re.IGNORECASE
    )
    feat_coaching_instability    = float(any(
        coaching_instability_pattern.search(t) for t in all_texts
    ))
    feat_ranking_mention_rate    = count_keyword_hits(all_texts, RANKING_KEYWORDS)
    feat_fatigue_signal          = count_keyword_hits(all_texts, FATIGUE_KEYWORDS)
    feat_home_away_context       = count_keyword_hits(all_texts, HOME_AWAY_KEYWORDS)

    # ─────────────────────────────────────────────────────────────────────────
    # [F] VOLUME / DATA CONFIDENCE
    # ─────────────────────────────────────────────────────────────────────────
    feat_total_articles          = min(total_articles / 35.0, 1.0)   # norm 0–1
    feat_source_diversity        = float(
        (len(gn_articles) > 0) + (len(espn_articles) > 0) +
        (len(injury_articles) > 0) + (len(lineup_articles) > 0)
    ) / 4.0
    # Overall data confidence: how much do we actually know about this team?
    feat_data_confidence         = (feat_total_articles * 0.5 +
                                    feat_source_diversity * 0.5)

    # ─────────────────────────────────────────────────────────────────────────
    # ASSEMBLE FEATURE DICT (all 27 features, clearly tagged)
    # ─────────────────────────────────────────────────────────────────────────
    features = {
        # Meta
        'team_name':                     team_name,

        # [A] Sentiment (7 features)
        'sent_overall':                  round(feat_sentiment_overall,     4),
        'sent_espn':                     round(feat_sentiment_espn,        4),
        'sent_google':                   round(feat_sentiment_google,      4),
        'sent_headlines':                round(feat_sentiment_headlines,   4),
        'sent_recent':                   round(feat_sentiment_recent,      4),
        'sent_pct_positive_articles':    round(feat_pct_positive_articles, 4),
        'sent_pct_negative_articles':    round(feat_pct_negative_articles, 4),

        # [B] Injury (5 features)
        'inj_mention_rate':              round(feat_injury_mention_rate,    4),
        'inj_severity_score':            round(feat_injury_severity_score,  4),
        'inj_article_count_norm':        round(feat_injury_article_count,   4),
        'inj_key_player_flag':           feat_key_player_injury_flag,
        'inj_sentiment':                 round(feat_injury_sentiment,       4),

        # [C] Lineup / Roster (5 features)
        'lineup_change_signal':          round(feat_lineup_change_signal,   4),
        'lineup_starter_mention_rate':   round(feat_starter_mention_rate,   4),
        'lineup_benched_signal':         round(feat_benched_signal,         4),
        'lineup_roster_instability':     round(feat_roster_instability,     4),
        'lineup_sentiment':              round(feat_lineup_sentiment,       4),

        # [D] Momentum / Form (6 features)
        'momentum_win_mention_rate':     round(feat_win_mention_rate,           4),
        'momentum_loss_mention_rate':    round(feat_loss_mention_rate,          4),
        'momentum_score':                round(feat_momentum_score,             4),
        'momentum_slump_score':          round(feat_slump_score,                4),
        'momentum_net':                  round(feat_net_momentum,               4),
        'momentum_win_loss_ratio':       round(feat_win_loss_coverage_ratio,    4),

        # [E] Context (5 features)
        'ctx_coaching_mention_rate':     round(feat_coaching_mention_rate,  4),
        'ctx_coaching_instability':      feat_coaching_instability,
        'ctx_ranking_mention_rate':      round(feat_ranking_mention_rate,   4),
        'ctx_fatigue_signal':            round(feat_fatigue_signal,         4),
        'ctx_home_away_context':         round(feat_home_away_context,      4),

        # [F] Data Quality (3 features)
        'data_total_articles_norm':      round(feat_total_articles,         4),
        'data_source_diversity':         round(feat_source_diversity,       4),
        'data_confidence':               round(feat_data_confidence,        4),
    }

    return features


print("✅ Cell 5 loaded — extract_team_features() defined.")
print("   27 features across 6 categories: Sentiment, Injury, Lineup,")
print("   Momentum, Context, Data Quality.")



✅ Cell 5 loaded — extract_team_features() defined.
   27 features across 6 categories: Sentiment, Injury, Lineup,
   Momentum, Context, Data Quality.


## Cell 6

In [7]:
# =============================================================================
# CELL 6 — Build Feature Matrix for All Teams Playing Today
# =============================================================================

import pandas as pd

def get_unique_teams(games_today):
    """
    Extracts unique team names from the merged game list (ESPN + CBS).
    Works entirely off team name strings — no IDs needed.
    """
    teams = {}
    for g in games_today:
        for side in ['home', 'away']:
            name = g[f'{side}_name']
            if name and name not in teams:
                teams[name] = {
                    'team_name': name,
                    'abbr':      g.get(f'{side}_abbr', name[:4].upper()),
                    'source':    g.get('source', 'unknown')
                }
    return teams


def build_feature_matrix(games_today):
    """
    Main orchestrator for Cells 4–6.
    For every unique team playing today:
      1. Fetches Google News + ESPN news articles by team name
      2. Extracts all 27 features
      3. Assembles into a single feature matrix DataFrame

    Returns:
      feature_matrix : pd.DataFrame  — one row per team, 27 feature columns
      team_lookup    : dict           — team_name → feature dict (for fast access)
    """
    unique_teams = get_unique_teams(games_today)
    total        = len(unique_teams)
    print(f"\n{'='*60}")
    print(f"  Building feature matrix for {total} teams...")
    print(f"  Sources: Google News RSS + ESPN News Search")
    print(f"  Features per team: 27 across 6 categories")
    print(f"{'='*60}\n")

    all_features = []
    team_lookup  = {}

    for i, (team_name, meta) in enumerate(unique_teams.items()):
        print(f"[{i+1:>3}/{total}] {team_name:<40}", end=' ')
        try:
            feats = extract_team_features(team_name)
            feats['data_source'] = meta['source']
            feats['abbr']        = meta['abbr']
            all_features.append(feats)
            team_lookup[team_name] = feats

            # Quick inline summary for visibility
            print(
                f"sent={feats['sent_overall']:+.3f}  "
                f"inj={feats['inj_mention_rate']:.2f}  "
                f"momentum={feats['momentum_net']:+.3f}  "
                f"conf={feats['data_confidence']:.2f}  "
                f"arts={round(feats['data_total_articles_norm']*35)}"
            )
        except Exception as e:
            print(f"  ❌ FAILED: {e}")
            # Insert zeroed-out row so the team still appears in the matrix
            blank = {k: 0.0 for k in [
                'sent_overall','sent_espn','sent_google','sent_headlines',
                'sent_recent','sent_pct_positive_articles','sent_pct_negative_articles',
                'inj_mention_rate','inj_severity_score','inj_article_count_norm',
                'inj_key_player_flag','inj_sentiment',
                'lineup_change_signal','lineup_starter_mention_rate',
                'lineup_benched_signal','lineup_roster_instability','lineup_sentiment',
                'momentum_win_mention_rate','momentum_loss_mention_rate',
                'momentum_score','momentum_slump_score','momentum_net',
                'momentum_win_loss_ratio',
                'ctx_coaching_mention_rate','ctx_coaching_instability',
                'ctx_ranking_mention_rate','ctx_fatigue_signal','ctx_home_away_context',
                'data_total_articles_norm','data_source_diversity','data_confidence'
            ]}
            blank['team_name']   = team_name
            blank['data_source'] = meta['source']
            blank['abbr']        = meta['abbr']
            all_features.append(blank)
            team_lookup[team_name] = blank

        time.sleep(0.4)   # polite throttle

    feature_matrix = pd.DataFrame(all_features)

    # Reorder columns: meta first, then by category
    meta_cols      = ['team_name', 'abbr', 'data_source']
    sentiment_cols = [c for c in feature_matrix.columns if c.startswith('sent_')]
    injury_cols    = [c for c in feature_matrix.columns if c.startswith('inj_')]
    lineup_cols    = [c for c in feature_matrix.columns if c.startswith('lineup_')]
    momentum_cols  = [c for c in feature_matrix.columns if c.startswith('momentum_')]
    context_cols   = [c for c in feature_matrix.columns if c.startswith('ctx_')]
    data_cols      = [c for c in feature_matrix.columns if c.startswith('data_')]

    ordered_cols = (meta_cols + sentiment_cols + injury_cols +
                    lineup_cols + momentum_cols + context_cols + data_cols)
    feature_matrix = feature_matrix[ordered_cols]

    return feature_matrix, team_lookup


# ── RUN ───────────────────────────────────────────────────────────────────────
feature_matrix, team_lookup = build_feature_matrix(games_today)

print(f"\n✅ Feature matrix complete: {feature_matrix.shape[0]} teams × {feature_matrix.shape[1]-3} features")



  Building feature matrix for 36 teams...
  Sources: Google News RSS + ESPN News Search
  Features per team: 27 across 6 categories

[  1/36] Morgan St.                                     📄 Fetched raw HTML (579,030 chars): Morgan State 90-83 South Carolina State (25 Feb, 2026) Final...
      📄 Fetched raw HTML (579,716 chars): Morgan State vs. South Carolina State (26 Feb, 2026) Live Sc...
      📄 Fetched raw HTML (583,315 chars): The Jersey Mike’s Jamaica Classic Returns to Montego Bay in ...
      📄 Fetched raw HTML (579,081 chars): Morgan State 128-53 Virginia-Lynchburg (20 Jan, 2026) Video ...
      📄 Fetched raw HTML (579,353 chars): Meet The NCAA’s 5-Foot-9 Scoring Machine - ABC News...
sent=+0.606  inj=0.00  momentum=+0.000  conf=0.24  arts=8
[  2/36] Norfolk St.                                    📄 Fetched raw HTML (579,738 chars): Norfolk St. vs Delaware State scores & predictions - Sofasco...
      📄 Fetched raw HTML (579,076 chars): These 2025 NCAA Men’s Tournament Teams 

In [8]:
feature_matrix.head()

,team_name,abbr,data_source,sent_overall,sent_espn,sent_google,sent_headlines,sent_recent,sent_pct_positive_articles,sent_pct_negative_articles,...,momentum_win_loss_ratio,ctx_coaching_mention_rate,ctx_coaching_instability,ctx_ranking_mention_rate,ctx_fatigue_signal,ctx_home_away_context,data_total_articles_norm,data_source_diversity,data_confidence,data_source
0,Morgan St.,MORGAN,unknown,0.6058,0.0,0.6058,0.0000,0.3877,0.6250,0.0000,...,1.0,0.7500,0.0,0.6250,0.6250,0.0,0.2286,0.25,0.2393,unknown
1,Norfolk St.,NORFLK,unknown,0.4174,0.0,0.4174,-0.0256,-0.0508,0.5455,0.0909,...,1.2,0.4545,0.0,0.6364,0.4545,0.0,0.3143,0.25,0.2821,unknown
2,Howard,HOW,unknown,0.2634,0.0,0.2634,0.0107,0.0405,0.4000,0.0500,...,1.4,0.3500,0.0,0.2500,0.2500,0.0,0.5714,0.25,0.4107,unknown
3,Coppin St.,COPPST,unknown,0.9657,0.0,0.9657,-0.4111,0.9657,1.0000,0.0000,...,1.0,1.0000,0.0,1.0000,1.0000,0.0,0.0571,0.25,0.1536,unknown
4,NC State,NCST,unknown,0.2313,0.0,0.2313,-0.0140,-0.0024,0.5500,0.2000,...,1.4,0.4000,0.0,0.3500,0.2500,0.0,0.5714,0.25,0.4107,unknown


# Feature Display

## Cell 7

In [9]:
'''# =============================================================================
# CELL 7 — Display Feature Matrix
# =============================================================================

def display_feature_matrix(feature_matrix):

    # Force all numeric columns to float to avoid mixed-type errors
    meta_cols    = ['team_name', 'abbr', 'data_source']
    numeric_cols = [c for c in feature_matrix.columns if c not in meta_cols]

    fm = feature_matrix.copy()
    for col in numeric_cols:
        fm[col] = pd.to_numeric(fm[col], errors='coerce').fillna(0.0)

    # Reset index to guarantee uniqueness — fixes Styler non-unique index error
    fm = fm.reset_index(drop=True)

    # ── Full feature matrix ───────────────────────────────────────────────────
    display(
        fm.style
        .background_gradient(subset=numeric_cols, cmap='RdYlGn', axis=0)
        .format({c: '{:.3f}' for c in numeric_cols})
        .set_caption("Feature Matrix — Green = High/Positive, Red = Low/Negative")
    )

    # ── Category summary ──────────────────────────────────────────────────────
    category_map = {
        '😊 Sentiment': [c for c in numeric_cols if c.startswith('sent_')],
        '🏥 Injury':    [c for c in numeric_cols if c.startswith('inj_')],
        '📋 Lineup':    [c for c in numeric_cols if c.startswith('lineup_')],
        '🔥 Momentum':  [c for c in numeric_cols if c.startswith('momentum_')],
        '🌍 Context':   [c for c in numeric_cols if c.startswith('ctx_')],
        '📊 Data Conf': [c for c in numeric_cols if c.startswith('data_')],
    }

    summary_data = {'Team': fm['team_name'].values}
    for label, cols in category_map.items():
        if cols:
            summary_data[label] = fm[cols].mean(axis=1).round(3).values

    summary_df = pd.DataFrame(summary_data).reset_index(drop=True)
    summary_numeric_cols = [c for c in summary_df.columns if c != 'Team']

    display(
        summary_df.style
        .background_gradient(subset=summary_numeric_cols, cmap='RdYlGn', axis=0)
        .format({c: '{:.3f}' for c in summary_numeric_cols})
        .set_caption("Category Averages")
    )

    return summary_df


summary_df = display_feature_matrix(feature_matrix)'''

'# =============================================================================\n# CELL 7 — Display Feature Matrix\n# =============================================================================\n\ndef display_feature_matrix(feature_matrix):\n\n    # Force all numeric columns to float to avoid mixed-type errors\n    meta_cols    = [\'team_name\', \'abbr\', \'data_source\']\n    numeric_cols = [c for c in feature_matrix.columns if c not in meta_cols]\n\n    fm = feature_matrix.copy()\n    for col in numeric_cols:\n        fm[col] = pd.to_numeric(fm[col], errors=\'coerce\').fillna(0.0)\n\n    # Reset index to guarantee uniqueness — fixes Styler non-unique index error\n    fm = fm.reset_index(drop=True)\n\n    # ── Full feature matrix ───────────────────────────────────────────────────\n    display(\n        fm.style\n        .background_gradient(subset=numeric_cols, cmap=\'RdYlGn\', axis=0)\n        .format({c: \'{:.3f}\' for c in numeric_cols})\n        .set_caption("Feature Matrix — 

# Google Drive Connection

## Cell 8

In [10]:
# =============================================================================
# CELL 8 — Load Training Data from Google Drive
# =============================================================================

import os
import pandas as pd
import numpy as np
from google.colab import drive

drive.mount('/content/drive', force_remount=False)

DRIVE_PATH    = '/content/drive/MyDrive/Colab Notebooks'
games_path    = os.path.join(DRIVE_PATH, 'ncaab_games_2.csv')
features_path = os.path.join(DRIVE_PATH, 'ncaab_team_features_2.csv')

games_df    = pd.read_csv(games_path)
features_df = pd.read_csv(features_path)

# ── META_COLS: all non-numeric identifier columns ─────────────────────────────
# Expand this list to cover any string column that should never be coerced
META_COLS = ['team_name', 'abbr', 'data_source']

# ── Force numeric on feature columns only ────────────────────────────────────
numeric_cols = [c for c in features_df.columns if c not in META_COLS]
for col in numeric_cols:
    features_df[col] = pd.to_numeric(features_df[col], errors='coerce').fillna(0.0)

# ── Build team lookup dict for training data ──────────────────────────────────
train_team_lookup = {row['team_name']: row.to_dict()
                     for _, row in features_df.iterrows()}

# ── fm_clean ──────────────────────────────────────────────────────────────────
def build_fm_clean(df):
    fm = df.copy().reset_index(drop=True)
    for col in fm.columns:
        if col in META_COLS:
            fm[col] = fm[col].astype(str)
            continue
        fm[col] = pd.to_numeric(fm[col], errors='coerce').fillna(0.0)
    return fm

try:
    fm_clean     = build_fm_clean(feature_matrix)
    fm_source    = "Cell 7 feature_matrix"
except NameError:
    fm_clean     = build_fm_clean(features_df)
    fm_source    = "features CSV fallback"

# ── Rebuild numeric_cols from fm_clean to be consistent ──────────────────────
numeric_cols = [c for c in fm_clean.columns if c not in META_COLS]

# ── Summary ───────────────────────────────────────────────────────────────────
print(f"✅ Cell 8 complete:")
print(f"   Games             : {games_df.shape[0]} rows × {games_df.shape[1]} cols")
print(f"   Features CSV      : {features_df.shape[0]} rows × {features_df.shape[1]} cols")
print(f"   Train team lookup : {len(train_team_lookup)} teams")
print(f"   fm_clean source   : {fm_source}")
print(f"   fm_clean          : {len(fm_clean)} teams × {fm_clean.shape[1]} cols")
print(f"   numeric_cols      : {len(numeric_cols)} features")

non_float = [(c, str(fm_clean[c].dtype)) for c in numeric_cols
             if fm_clean[c].dtype not in ['float64', 'float32', 'int64']]
if non_float:
    print(f"\n  ⚠️  Non-numeric columns still present:")
    for col, dtype in non_float:
        print(f"     {col}: {dtype}  sample={fm_clean[col].iloc[0]}")
else:
    print(f"   ✅ All {len(numeric_cols)} feature columns confirmed numeric")

display(games_df.head())
display(fm_clean.head())

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
✅ Cell 8 complete:
   Games             : 691 rows × 12 cols
   Features CSV      : 365 rows × 32 cols
   Train team lookup : 365 teams
   fm_clean source   : Cell 7 feature_matrix
   fm_clean          : 36 teams × 35 cols
   numeric_cols      : 31 features
   ✅ All 31 feature columns confirmed numeric


,date,away_name,away_score,home_name,home_score,status,home_team_won,winner_name,winner_score,loser_name,loser_score,score_diff
0,2026-02-16,SE Louisiana,53,East Texas A&M,70,Final,1,East Texas A&M,70,SE Louisiana,53,17
1,2026-02-16,Coppin St.,59,SC State,57,Final,0,Coppin St.,59,SC State,57,-2
2,2026-02-16,Louisiana,72,Old Dominion,83,Final,1,Old Dominion,83,Louisiana,72,11
3,2026-02-16,Colgate,58,Boston U.,85,Final,1,Boston U.,85,Colgate,58,27
4,2026-02-16,Bethune-Cook.,86,Jackson St.,91,Final,1,Jackson St.,91,Bethune-Cook.,86,5


,team_name,abbr,data_source,sent_overall,sent_espn,sent_google,sent_headlines,sent_recent,sent_pct_positive_articles,sent_pct_negative_articles,...,momentum_win_loss_ratio,ctx_coaching_mention_rate,ctx_coaching_instability,ctx_ranking_mention_rate,ctx_fatigue_signal,ctx_home_away_context,data_total_articles_norm,data_source_diversity,data_confidence,data_source
0,Morgan St.,MORGAN,unknown,0.6058,0.0,0.6058,0.0000,0.3877,0.6250,0.0000,...,1.0,0.7500,0.0,0.6250,0.6250,0.0,0.2286,0.25,0.2393,unknown
1,Norfolk St.,NORFLK,unknown,0.4174,0.0,0.4174,-0.0256,-0.0508,0.5455,0.0909,...,1.2,0.4545,0.0,0.6364,0.4545,0.0,0.3143,0.25,0.2821,unknown
2,Howard,HOW,unknown,0.2634,0.0,0.2634,0.0107,0.0405,0.4000,0.0500,...,1.4,0.3500,0.0,0.2500,0.2500,0.0,0.5714,0.25,0.4107,unknown
3,Coppin St.,COPPST,unknown,0.9657,0.0,0.9657,-0.4111,0.9657,1.0000,0.0000,...,1.0,1.0000,0.0,1.0000,1.0000,0.0,0.0571,0.25,0.1536,unknown
4,NC State,NCST,unknown,0.2313,0.0,0.2313,-0.0140,-0.0024,0.5500,0.2000,...,1.4,0.4000,0.0,0.3500,0.2500,0.0,0.5714,0.25,0.4107,unknown


# Machine Learning

## Cell 9

In [11]:
# =============================================================================
# CELL 9 — Build Training Dataset (X = team feature differentials, Y = score diff)
# =============================================================================

META_COLS    = ['team_name']
numeric_cols = [c for c in features_df.columns if c not in META_COLS]

def get_feature_vector(team_name, lookup, cols):
    if team_name in lookup:
        return np.array([float(lookup[team_name].get(c, 0.0)) for c in cols])
    return np.zeros(len(cols))

def build_matchup_vector(home_name, away_name, lookup, cols):
    """
    Builds a feature vector for a matchup:
      - home team features
      - away team features
      - differential (home - away)
      - is_home binary flag (1.0 = home perspective, 0.0 = away perspective)
    """
    home_vec = get_feature_vector(home_name, lookup, cols)
    away_vec = get_feature_vector(away_name, lookup, cols)
    diff_vec = home_vec - away_vec
    is_home  = np.array([1.0])
    return np.concatenate([home_vec, away_vec, diff_vec, is_home])

# ── Feature names (must match build_matchup_vector exactly) ───────────────────
feature_names = (
    [f"home_{c}" for c in numeric_cols] +
    [f"away_{c}" for c in numeric_cols] +
    [f"diff_{c}" for c in numeric_cols] +
    ["is_home"]
)
is_home_idx = feature_names.index("is_home")
print(f"✅ Feature names built — {len(feature_names)} features, is_home at index {is_home_idx}")

# ── Inspect raw CSV before any filtering ─────────────────────────────────────
print("\nRaw games_df dtypes:")
print(games_df.dtypes)
print(f"\nRaw games_df shape: {games_df.shape}")
print(f"\nFirst 10 rows of key columns:")
print(games_df[['home_name','away_name','home_score','away_score','home_team_won']].head(10).to_string())
print(f"\nNull counts per column:")
print(games_df[['home_name','away_name','home_score','away_score','home_team_won']].isnull().sum())
print(f"\nUnique values in home_team_won: {games_df['home_team_won'].unique()}")
print(f"Unique values in home_score sample: {games_df['home_score'].unique()[:10]}")

# ── Force numeric conversion before dropna ────────────────────────────────────
games_df['home_score']    = pd.to_numeric(games_df['home_score'],    errors='coerce')
games_df['away_score']    = pd.to_numeric(games_df['away_score'],    errors='coerce')
games_df['home_team_won'] = pd.to_numeric(games_df['home_team_won'], errors='coerce')

print(f"\nAfter to_numeric conversion:")
print(f"   Null home_score    : {games_df['home_score'].isnull().sum()}")
print(f"   Null away_score    : {games_df['away_score'].isnull().sum()}")
print(f"   Null home_team_won : {games_df['home_team_won'].isnull().sum()}")

# ── Only keep rows where game is final ────────────────────────────────────────
final_games = games_df[games_df['home_team_won'].notna()].copy()
final_games['score_diff'] = final_games['home_score'].astype(float) - \
                             final_games['away_score'].astype(float)

print(f"\n   Total rows in games CSV : {len(games_df)}")
print(f"   Final games found       : {len(final_games)}")

if len(final_games) == 0:
    print("\n❌ Still zero final games after numeric conversion.")
    print("   Printing ALL raw rows to inspect:")
    print(games_df.to_string())
    raise ValueError("No final games found in CSV — check Cell 6 saved scores correctly.")

# ── Build training arrays ─────────────────────────────────────────────────────
# For every game we build TWO rows:
#   Forward:  home team as "team A" → is_home=1.0, score_diff = home - away
#   Mirrored: away team as "team A" → is_home=0.0, score_diff = away - home
# This balances the dataset 50/50 and prevents the model from learning
# "home team always wins" simply due to label imbalance.

X_rows, y_rows, skipped = [], [], []

for _, row in final_games.iterrows():
    home = str(row['home_name']).strip()
    away = str(row['away_name']).strip()

    home_found = home in train_team_lookup
    away_found = away in train_team_lookup

    if not home_found or not away_found:
        skipped.append(f"{away} @ {home}  "
                       f"[home_found={home_found}, away_found={away_found}]")
        continue

    score_diff = float(row['score_diff'])

    # ── Forward: home team perspective ────────────────────────────────────────
    vec_home = build_matchup_vector(home, away, train_team_lookup, numeric_cols)
    X_rows.append(vec_home)
    y_rows.append(score_diff)

    # ── Mirrored: away team perspective ───────────────────────────────────────
    home_vec = get_feature_vector(home, train_team_lookup, numeric_cols)
    away_vec = get_feature_vector(away, train_team_lookup, numeric_cols)
    diff_vec = away_vec - home_vec    # flipped differential
    is_away  = np.array([0.0])        # binary: 0 = away perspective
    vec_away = np.concatenate([away_vec, home_vec, diff_vec, is_away])
    X_rows.append(vec_away)
    y_rows.append(-score_diff)        # flipped: positive = away won

if not X_rows:
    print(f"\n❌ No training samples built after finding {len(final_games)} final games.")
    print(f"   Skipped ({len(skipped)}):")
    for s in skipped[:20]:
        print(f"     • {s}")
    raise ValueError("Training set empty — check team name alignment.")

X_train = np.array(X_rows, dtype=np.float32)
y_train = np.array(y_rows, dtype=np.float32)

# ── Sanity check: feature_names length must match X_train columns ─────────────
assert X_train.shape[1] == len(feature_names), (
    f"❌ Mismatch: X_train has {X_train.shape[1]} columns "
    f"but feature_names has {len(feature_names)} entries. "
    f"Check build_matchup_vector() includes is_home."
)

# ── Class balance diagnostics ─────────────────────────────────────────────────
home_wins = int(np.sum(y_train > 0))
away_wins = int(np.sum(y_train < 0))
total     = len(y_train)
home_rate = home_wins / total

'''print(f"\n✅ Training set built:")
print(f"   Samples  : {X_train.shape[0]}  ({len(final_games)} games × 2 perspectives)")
print(f"   Features : {X_train.shape[1]}  (home + away + differential + is_home)")
print(f"   Y range  : {y_train.min():.1f} to {y_train.max():.1f} points")
print(f"   is_home index : {is_home_idx}")
print(f"\n📊 Label balance check:")
print(f"   Home-perspective samples : {home_wins}  ({home_wins/total:.1%})")
print(f"   Away-perspective samples : {away_wins}  ({away_wins/total:.1%})")'''
if abs(home_rate - 0.5) < 0.02:
    print(f"   ✅ Dataset is balanced (expected ~50% since we mirrored all games)")
else:
    print(f"   ⚠️  Imbalance detected ({home_rate:.1%} home) — check mirroring logic above")

if skipped:
    print(f"\n   Skipped : {len(skipped)} games (team not found in features lookup)")
    for s in skipped[:10]:
        print(f"     • {s}")

✅ Feature names built — 94 features, is_home at index 93

Raw games_df dtypes:
date             object
away_name        object
away_score        int64
home_name        object
home_score        int64
status           object
home_team_won     int64
winner_name      object
winner_score      int64
loser_name       object
loser_score       int64
score_diff        int64
dtype: object

Raw games_df shape: (691, 12)

First 10 rows of key columns:
        home_name        away_name  home_score  away_score  home_team_won
0  East Texas A&M     SE Louisiana          70          53              1
1        SC State       Coppin St.          57          59              0
2    Old Dominion        Louisiana          83          72              1
3       Boston U.          Colgate          85          58              1
4     Jackson St.    Bethune-Cook.          91          86              1
5            Duke         Syracuse         101          64              1
6    Prairie View    Grambling St.     

## Cell 10

In [12]:
# =============================================================================
# CELL 10 — Train 5 ML Models (with model caching — skips retraining if saved)
# =============================================================================

from sklearn.linear_model       import Ridge, LogisticRegression
from sklearn.ensemble           import GradientBoostingRegressor
from sklearn.preprocessing      import StandardScaler
from sklearn.model_selection    import cross_val_score, StratifiedKFold
from sklearn.utils.class_weight import compute_class_weight
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
import numpy as np
import joblib
import os
import warnings
warnings.filterwarnings('ignore')

# =============================================================================
# ── CACHE CONTROL ─────────────────────────────────────────────────────────────
#
#   USE_CACHE     = True  → use saved models if they exist (default)
#   USE_CACHE     = False → always train from scratch, never read or write cache
#
#   FORCE_RETRAIN = True  → retrain from scratch BUT save the new models to cache
#   FORCE_RETRAIN = False → normal behavior (load if available, train if not)
#
#   Behavior matrix:
#   ┌─────────────┬───────────────┬──────────────────────────────────────────┐
#   │ USE_CACHE   │ FORCE_RETRAIN │ Result                                   │
#   ├─────────────┼───────────────┼──────────────────────────────────────────┤
#   │ True        │ False         │ Load from cache if exists, else train    │
#   │ True        │ True          │ Always retrain and overwrite cache        │
#   │ False       │ False         │ Always train, never read or write cache  │
#   │ False       │ True          │ Same as above (USE_CACHE=False wins)     │
#   └─────────────┴───────────────┴──────────────────────────────────────────┘
# =============================================================================
USE_CACHE     = True
FORCE_RETRAIN = True

# =============================================================================
# ── REGULARIZATION TUNING — adjust based on Home Bias Audit results ───────────
#
#   If audit shows 🔴 OVERFIT HOME (>72%) → increase values (more regularization)
#   If audit shows 🟡 undervaluing (<50%) → decrease values (less regularization)
#   Target range: 55–62% home win rate in audit
#
#   Current tuning: relaxed for larger dataset (more data = less overfit risk)
# =============================================================================
RIDGE_ALPHA        = 20.0    # was 50.0 — relaxed for larger dataset
GB_N_ESTIMATORS    = 200     # was 100  — more trees now data supports it
GB_LEARNING_RATE   = 0.05    # unchanged
GB_MAX_DEPTH       = 4       # was 3    — slightly deeper trees
GB_MIN_SAMPLES     = 3       # was 5    — looser split constraint
GB_SUBSAMPLE       = 0.85    # was 0.8  — slightly more data per tree
LOGISTIC_C         = 0.5     # was 0.1  — less penalty, more signal allowed
NN_L2              = 0.01    # was 0.05 — much less L2 regularization
NN_DROPOUT         = 0.05     # was 0.5  — less dropout
NN_HIDDEN_1        = 128      # was 32   — wider first layer
NN_HIDDEN_2        = 64      # was 16   — wider second layer
NN_LEARNING_RATE   = 0.0003  # was 0.0003 — slightly faster learning
NN_BATCH_SIZE      = 32      # was 64   — smaller batch = more gradient updates

MODEL_CACHE_DIR = "/content/drive/MyDrive/ncaab_model_cache"
os.makedirs(MODEL_CACHE_DIR, exist_ok=True)

# ── Cache file paths ──────────────────────────────────────────────────────────
PATH_SCALER   = os.path.join(MODEL_CACHE_DIR, "scaler.joblib")
PATH_RIDGE    = os.path.join(MODEL_CACHE_DIR, "ridge_model.joblib")
PATH_GB       = os.path.join(MODEL_CACHE_DIR, "gb_model.joblib")
PATH_LOGISTIC = os.path.join(MODEL_CACHE_DIR, "logistic_model.joblib")
PATH_NN_REG   = os.path.join(MODEL_CACHE_DIR, "nn_regressor.keras")
PATH_NN_CLS   = os.path.join(MODEL_CACHE_DIR, "nn_classifier.keras")
PATH_METADATA = os.path.join(MODEL_CACHE_DIR, "metadata.joblib")

ALL_CACHE_FILES = [PATH_SCALER, PATH_RIDGE, PATH_GB,
                   PATH_LOGISTIC, PATH_NN_REG, PATH_NN_CLS]

def cache_exists():
    return all(os.path.exists(p) for p in ALL_CACHE_FILES)

def save_to_cache(scaler, lr_model, sgd_model, logistic_model,
                  nn_reg_model, nn_cls_model, metadata_dict):
    from datetime import date
    print(f"\n💾 Saving models to cache: {MODEL_CACHE_DIR}")
    joblib.dump(scaler,         PATH_SCALER)
    joblib.dump(lr_model,       PATH_RIDGE)
    joblib.dump(sgd_model,      PATH_GB)
    joblib.dump(logistic_model, PATH_LOGISTIC)
    nn_reg_model.save(PATH_NN_REG)
    nn_cls_model.save(PATH_NN_CLS)
    joblib.dump(metadata_dict,  PATH_METADATA)
    print(f"   ✅ All models saved — trained on {metadata_dict.get('training_date', '?')} "
          f"with {metadata_dict.get('n_samples', '?')} samples")

# ── Ensure is_home_idx is always defined regardless of run order ───────────────
META_COLS     = ['team_name']
numeric_cols  = [c for c in features_df.columns if c not in META_COLS]
feature_names = (
    [f"home_{c}" for c in numeric_cols] +
    [f"away_{c}" for c in numeric_cols] +
    [f"diff_{c}" for c in numeric_cols] +
    ["is_home"]
)
is_home_idx = feature_names.index("is_home")
print(f"✅ is_home_idx = {is_home_idx}  (total features: {len(feature_names)})")
print(f"   USE_CACHE={USE_CACHE}  |  FORCE_RETRAIN={FORCE_RETRAIN}")
print(f"   Dataset size: {X_train.shape[0]} samples\n")


# =============================================================================
# DECIDE: LOAD FROM CACHE or TRAIN FROM SCRATCH
# =============================================================================
should_load = USE_CACHE and cache_exists() and not FORCE_RETRAIN
should_save = USE_CACHE

if should_load:
    # =========================================================================
    # LOAD FROM CACHE
    # =========================================================================
    print("💾 Saved models found — loading from cache\n"
          "   (set FORCE_RETRAIN=True to retrain, USE_CACHE=False to disable caching)\n")

    scaler         = joblib.load(PATH_SCALER)
    lr_model       = joblib.load(PATH_RIDGE)
    sgd_model      = joblib.load(PATH_GB)
    logistic_model = joblib.load(PATH_LOGISTIC)
    nn_reg_model   = keras.models.load_model(PATH_NN_REG)
    nn_cls_model   = keras.models.load_model(PATH_NN_CLS)
    metadata       = joblib.load(PATH_METADATA)

    y_train_binary    = (y_train > 0).astype(int)   # always recompute from current data
    class_weight_dict = metadata['class_weight_dict']
    lr_cv             = metadata['lr_cv']
    sgd_cv            = metadata['sgd_cv']
    logistic_cv_acc   = metadata['logistic_cv_acc']
    logistic_cv_auc   = metadata['logistic_cv_auc']
    cls_val_acc       = metadata['cls_val_acc']
    cls_val_auc       = metadata['cls_val_auc']
    reg_history       = metadata['reg_history']
    history           = reg_history

    X_scaled       = scaler.transform(X_train)      # always recompute from current data
    X_scaled_train = X_scaled.copy()
    X_scaled_train[:, is_home_idx] = 0.0

    print(f"   ✅ All 5 models loaded from cache")
    print(f"\n   Trained on  : {metadata.get('training_date', 'unknown')}")
    print(f"   Cached with : {metadata.get('n_samples', '?')} samples  |  "
          f"{metadata.get('n_features', '?')} features")
    print(f"   Current data: {X_train.shape[0]} samples  |  {X_train.shape[1]} features")

    if X_train.shape[0] != metadata.get('n_samples'):
        print(f"\n   ⚠️  Dataset size changed since last training "
              f"({metadata.get('n_samples')} → {X_train.shape[0]} samples)")
        print(f"       Consider setting FORCE_RETRAIN=True to retrain on the full dataset")

else:
    # =========================================================================
    # TRAIN FROM SCRATCH
    # =========================================================================
    if not USE_CACHE:
        print("ℹ️  USE_CACHE=False — training from scratch (cache will not be read or written)\n")
    elif FORCE_RETRAIN:
        print("⚠️  FORCE_RETRAIN=True — retraining from scratch and overwriting cache\n")
    else:
        print("🆕 No saved models found — training from scratch\n")

    # ── Shared scaler ─────────────────────────────────────────────────────────
    scaler         = StandardScaler()
    X_scaled       = scaler.fit_transform(X_train)
    X_scaled_train = X_scaled.copy()
    X_scaled_train[:, is_home_idx] = 0.0
    print("ℹ️  is_home zeroed out in training matrix to prevent shortcut learning.\n")

    # ── Binary labels ─────────────────────────────────────────────────────────
    y_train_binary = (y_train > 0).astype(int)
    home_win_rate  = y_train_binary.mean()

    print(f"   Score diff range  : {y_train.min():.1f} to {y_train.max():.1f}")
    print(f"   Raw home win rate : {home_win_rate:.1%}")

    home_rows_mask = X_train[:, is_home_idx] == 1.0
    away_rows_mask = X_train[:, is_home_idx] == 0.0
    print(f"   Home-perspective rows : {home_rows_mask.sum()}")
    print(f"   Away-perspective rows : {away_rows_mask.sum()}")

    if abs(home_win_rate - 0.5) > 0.05:
        print(f"   ⚠️  Label imbalance detected ({home_win_rate:.1%}) — check Cell 9 mirroring")
    else:
        print(f"   ✅ Labels balanced at {home_win_rate:.1%}")
    print()

    # ── Class weights ─────────────────────────────────────────────────────────
    class_weights_array = compute_class_weight(
        class_weight='balanced', classes=np.array([0, 1]), y=y_train_binary
    )
    class_weight_dict = {0: float(class_weights_array[0]),
                         1: float(class_weights_array[1])}
    print(f"   Class weights → away(0): {class_weight_dict[0]:.3f}  "
          f"| home(1): {class_weight_dict[1]:.3f}\n")

    N_FOLDS = min(5, len(X_train) // 2)
    skf     = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=42)

    # ── Model 1: Ridge ────────────────────────────────────────────────────────
    print(f"📐 Training Model 1: Ridge Regression  [alpha={RIDGE_ALPHA}]...")
    lr_model = Ridge(alpha=RIDGE_ALPHA)
    lr_model.fit(X_scaled_train, y_train)
    lr_cv = cross_val_score(lr_model, X_scaled_train, y_train,
                             cv=N_FOLDS, scoring='neg_mean_absolute_error')
    print(f"   CV MAE: {-lr_cv.mean():.2f} ± {lr_cv.std():.2f} points")
    print(f"   is_home coefficient : {lr_model.coef_[is_home_idx]:+.4f}")

    # ── Model 2: Gradient Boosting ────────────────────────────────────────────
    print(f"\n🌲 Training Model 2: Gradient Boosting  "
          f"[n={GB_N_ESTIMATORS}, depth={GB_MAX_DEPTH}, lr={GB_LEARNING_RATE}]...")
    sgd_model = GradientBoostingRegressor(
        n_estimators  = GB_N_ESTIMATORS,
        learning_rate = GB_LEARNING_RATE,
        max_depth     = GB_MAX_DEPTH,
        min_samples_leaf = GB_MIN_SAMPLES,
        subsample     = GB_SUBSAMPLE,
        max_features  = 0.7,
        random_state  = 42
    )
    sgd_model.fit(X_scaled_train, y_train)
    sgd_cv = cross_val_score(sgd_model, X_scaled_train, y_train,
                              cv=N_FOLDS, scoring='neg_mean_absolute_error')
    print(f"   CV MAE: {-sgd_cv.mean():.2f} ± {sgd_cv.std():.2f} points")
    top5_idx = np.argsort(sgd_model.feature_importances_)[::-1][:5]
    print(f"   Top 5 features:")
    for i in top5_idx:
        print(f"     {feature_names[i]:<35} importance: {sgd_model.feature_importances_[i]:.4f}")

    # ── Model 3: Neural Network Regressor ─────────────────────────────────────
    print(f"\n🧠 Training Model 3: NN Regressor  "
          f"[{NN_HIDDEN_1}→{NN_HIDDEN_2}, dropout={NN_DROPOUT}, l2={NN_L2}]...")
    nn_reg_model = keras.Sequential([
        layers.Input(shape=(X_scaled_train.shape[1],)),
        layers.Dense(NN_HIDDEN_1, activation='relu',
                     kernel_regularizer=keras.regularizers.l2(NN_L2)),
        layers.BatchNormalization(),
        layers.Dropout(NN_DROPOUT),
        layers.Dense(NN_HIDDEN_2, activation='relu',
                     kernel_regularizer=keras.regularizers.l2(NN_L2)),
        layers.BatchNormalization(),
        layers.Dropout(NN_DROPOUT / 2),   # second layer gets half dropout
        layers.Dense(1, activation='linear')
    ], name='nn_regressor')
    nn_reg_model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=NN_LEARNING_RATE),
        loss='mse', metrics=['mae']
    )
    reg_history = nn_reg_model.fit(
        X_scaled_train, y_train,
        epochs=200, batch_size=NN_BATCH_SIZE,
        validation_split=0.20, verbose=0,
        callbacks=[
            keras.callbacks.EarlyStopping(patience=20, restore_best_weights=True,
                                          monitor='val_loss'),
            keras.callbacks.ReduceLROnPlateau(patience=10, factor=0.5, min_lr=1e-5)
        ]
    )
    reg_train_mae = min(reg_history.history.get('mae',     [float('inf')]))
    reg_val_mae   = min(reg_history.history.get('val_mae', [float('inf')]))
    overfit_gap   = reg_train_mae - reg_val_mae
    print(f"   Train MAE : {reg_train_mae:.2f}  |  Val MAE : {reg_val_mae:.2f}  |  "
          f"Gap : {overfit_gap:+.2f}  "
          f"({'⚠️  overfitting' if abs(overfit_gap) > 3.0 else '✅ healthy'})")

    # ── Model 4: Logistic Regression ──────────────────────────────────────────
    print(f"\n📊 Training Model 4: Logistic Regression  [C={LOGISTIC_C}]...")
    logistic_model = LogisticRegression(
        penalty='elasticnet', solver='saga', l1_ratio=0.5,
        C=LOGISTIC_C, max_iter=2000, class_weight='balanced', random_state=42
    )
    logistic_model.fit(X_scaled_train, y_train_binary)
    logistic_cv_acc = cross_val_score(logistic_model, X_scaled_train, y_train_binary,
                                       cv=skf, scoring='accuracy')
    logistic_cv_auc = cross_val_score(logistic_model, X_scaled_train, y_train_binary,
                                       cv=skf, scoring='roc_auc')
    print(f"   CV Accuracy : {logistic_cv_acc.mean():.1%} ± {logistic_cv_acc.std():.1%}")
    print(f"   CV ROC-AUC  : {logistic_cv_auc.mean():.3f} ± {logistic_cv_auc.std():.3f}")

    # ── Model 5: Neural Network Classifier ────────────────────────────────────
    print(f"\n🤖 Training Model 5: NN Classifier  "
          f"[{NN_HIDDEN_1}→{NN_HIDDEN_2}, dropout={NN_DROPOUT}, l2={NN_L2}]...")
    nn_cls_model = keras.Sequential([
        layers.Input(shape=(X_scaled_train.shape[1],)),
        layers.Dense(NN_HIDDEN_1, activation='relu',
                     kernel_regularizer=keras.regularizers.l2(NN_L2)),
        layers.BatchNormalization(),
        layers.Dropout(NN_DROPOUT),
        layers.Dense(NN_HIDDEN_2, activation='relu',
                     kernel_regularizer=keras.regularizers.l2(NN_L2)),
        layers.BatchNormalization(),
        layers.Dropout(NN_DROPOUT / 2),
        layers.Dense(1, activation='sigmoid')
    ], name='nn_classifier')
    nn_cls_model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=NN_LEARNING_RATE),
        loss='binary_crossentropy',
        metrics=['accuracy', tf.keras.metrics.AUC(name='auc')]
    )
    cls_history = nn_cls_model.fit(
        X_scaled_train, y_train_binary,
        epochs=200, batch_size=NN_BATCH_SIZE,
        validation_split=0.20, verbose=0,
        class_weight=class_weight_dict,
        callbacks=[
            keras.callbacks.EarlyStopping(patience=20, restore_best_weights=True,
                                          monitor='val_loss'),
            keras.callbacks.ReduceLROnPlateau(patience=10, factor=0.5, min_lr=1e-5)
        ]
    )
    cls_train_acc = max(cls_history.history.get('accuracy',     [0]))
    cls_val_acc   = max(cls_history.history.get('val_accuracy', [0]))
    cls_val_auc   = max(cls_history.history.get('val_auc',      [0]))
    cls_gap       = cls_train_acc - cls_val_acc
    print(f"   Train Acc : {cls_train_acc:.1%}  |  Val Acc : {cls_val_acc:.1%}  |  "
          f"Gap : {cls_gap:+.1%}  "
          f"({'⚠️  overfitting' if cls_gap > 0.08 else '✅ healthy'})")
    print(f"   Best Val AUC : {cls_val_auc:.3f}")

    history = reg_history

    # ── Save to cache only if USE_CACHE is True ───────────────────────────────
    if should_save:
        from datetime import date
        save_to_cache(
            scaler, lr_model, sgd_model, logistic_model,
            nn_reg_model, nn_cls_model,
            metadata_dict={
                'y_train_binary':    y_train_binary,
                'class_weight_dict': class_weight_dict,
                'lr_cv':             lr_cv,
                'sgd_cv':            sgd_cv,
                'logistic_cv_acc':   logistic_cv_acc,
                'logistic_cv_auc':   logistic_cv_auc,
                'cls_val_acc':       cls_val_acc,
                'cls_val_auc':       cls_val_auc,
                'reg_history':       reg_history,
                'training_date':     str(date.today()),
                'n_samples':         X_train.shape[0],
                'n_features':        X_train.shape[1],
                # Save tuning params so you know what settings produced the cache
                'ridge_alpha':       RIDGE_ALPHA,
                'logistic_c':        LOGISTIC_C,
                'nn_l2':             NN_L2,
                'nn_dropout':        NN_DROPOUT,
            }
        )
    else:
        print(f"\n   ℹ️  USE_CACHE=False — models not saved to disk")


# =============================================================================
# POST-TRAINING HOME BIAS AUDIT (runs regardless of load vs train)
# Tune the REGULARIZATION TUNING constants above until all models show ✅
# =============================================================================
print("\n" + "="*60)
print("  📊 Home Bias Audit (target range: 55–62% for NCAA)")
print("="*60)

home_rows_mask = X_train[:, is_home_idx] == 1.0
lr_home_rate   = np.mean(lr_model.predict(X_scaled_train)[home_rows_mask]  > 0)
sgd_home_rate  = np.mean(sgd_model.predict(X_scaled_train)[home_rows_mask] > 0)
nn_reg_preds   = nn_reg_model.predict(X_scaled_train, verbose=0).flatten()
nn_reg_rate    = np.mean(nn_reg_preds[home_rows_mask] > 0)
log_home_rate  = np.mean(
    logistic_model.predict_proba(X_scaled_train)[home_rows_mask, 1] > 0.5)
nn_cls_preds   = nn_cls_model.predict(X_scaled_train, verbose=0).flatten()
nn_cls_rate    = np.mean(nn_cls_preds[home_rows_mask] > 0.5)

all_in_range = True
for name, rate in [("Ridge",          lr_home_rate),
                   ("Grad Boost",      sgd_home_rate),
                   ("NN Regressor",    nn_reg_rate),
                   ("Logistic",        log_home_rate),
                   ("NN Classifier",   nn_cls_rate)]:
    if rate > 0.72:
        flag = "🔴 OVERFIT HOME — increase regularization constant"
        all_in_range = False
    elif rate > 0.62:
        flag = "🟡 slightly high — consider small regularization increase"
        all_in_range = False
    elif rate < 0.50:
        flag = "🟡 undervaluing home court — decrease regularization constant"
        all_in_range = False
    else:
        flag = "✅ well calibrated"
    print(f"   {name:<16} home win rate: {rate:.1%}  {flag}")

print()
if all_in_range:
    print("  ✅ All models well calibrated — no tuning needed")
else:
    print("  ℹ️  Adjust the REGULARIZATION TUNING constants at the top of this cell")
    print("      then set FORCE_RETRAIN=True and rerun")

print("\n" + "="*60)
print("  ✅ All 5 models ready:")
print("     Models 1–3 → predict SCORE DIFFERENTIAL (regression)")
print("     Models 4–5 → predict WIN/LOSS probability (classification)")
print("="*60)

✅ is_home_idx = 93  (total features: 94)
   USE_CACHE=True  |  FORCE_RETRAIN=False
   Dataset size: 1382 samples

💾 Saved models found — loading from cache
   (set FORCE_RETRAIN=True to retrain, USE_CACHE=False to disable caching)

   ✅ All 5 models loaded from cache

   Trained on  : 2026-02-26
   Cached with : 786 samples  |  94 features
   Current data: 1382 samples  |  94 features

   ⚠️  Dataset size changed since last training (786 → 1382 samples)
       Consider setting FORCE_RETRAIN=True to retrain on the full dataset

  📊 Home Bias Audit (target range: 55–62% for NCAA)
   Ridge            home win rate: 51.1%  ✅ well calibrated
   Grad Boost       home win rate: 49.1%  🟡 undervaluing home court — decrease regularization constant
   NN Regressor     home win rate: 41.2%  🟡 undervaluing home court — decrease regularization constant
   Logistic         home win rate: 47.9%  🟡 undervaluing home court — decrease regularization constant
   NN Classifier    home win rate: 45.9%  🟡 un

## Cell 10B

In [13]:
# =============================================================================
# CELL 10B — Model visualization
# =============================================================================
# =============================================================================
# MODEL PERFORMANCE SUMMARY TABLE
# =============================================================================
from sklearn.metrics import accuracy_score, roc_auc_score, mean_absolute_error
import pandas as pd

summary_rows = []

# ── Regression models: derive accuracy by checking if sign of prediction matches y ──
lr_preds_train   = lr_model.predict(X_scaled)
sgd_preds_train  = sgd_model.predict(X_scaled)
nn_reg_preds_all = nn_reg_model.predict(X_scaled, verbose=0).flatten()

for name, preds, cv_scores in [
    ("Ridge Regression",    lr_preds_train,   lr_cv),
    ("SGD Regressor",       sgd_preds_train,  sgd_cv),
    ("NN Regressor",        nn_reg_preds_all, None),
]:
    predicted_winner = (preds > 0).astype(int)
    acc   = accuracy_score(y_train_binary, predicted_winner)
    try:
        auc = roc_auc_score(y_train_binary, preds)
    except Exception:
        auc = float('nan')
    mae   = mean_absolute_error(y_train, preds)
    cv_mae = f"{-cv_scores.mean():.2f} ± {cv_scores.std():.2f}" if cv_scores is not None else "N/A"
    summary_rows.append({
        'Model':         name,
        'Type':          'Regression',
        'Train Acc':     f"{acc:.1%}",
        'CV MAE':        cv_mae,
        'Train MAE':     f"{mae:.2f} pts",
        'ROC-AUC':       f"{auc:.3f}",
        'Val Acc':       '—',
        'Val AUC':       '—',
    })

# ── Classification models ─────────────────────────────────────────────────────
log_preds_prob  = logistic_model.predict_proba(X_scaled)[:, 1]
log_preds_class = (log_preds_prob > 0.5).astype(int)
log_train_acc   = accuracy_score(y_train_binary, log_preds_class)
log_train_auc   = roc_auc_score(y_train_binary, log_preds_prob)

nn_cls_preds_all   = nn_cls_model.predict(X_scaled, verbose=0).flatten()
nn_cls_preds_class = (nn_cls_preds_all > 0.5).astype(int)
nn_cls_train_acc   = accuracy_score(y_train_binary, nn_cls_preds_class)
nn_cls_train_auc   = roc_auc_score(y_train_binary, nn_cls_preds_all)

summary_rows.append({
    'Model':     'Logistic Regression',
    'Type':      'Classifier',
    'Train Acc': f"{log_train_acc:.1%}",
    'CV MAE':    '—',
    'Train MAE': '—',
    'ROC-AUC':   f"{log_train_auc:.3f}",
    'Val Acc':   f"{logistic_cv_acc.mean():.1%} ± {logistic_cv_acc.std():.1%}",
    'Val AUC':   f"{logistic_cv_auc.mean():.3f} ± {logistic_cv_auc.std():.3f}",
})

summary_rows.append({
    'Model':     'NN Classifier',
    'Type':      'Classifier',
    'Train Acc': f"{nn_cls_train_acc:.1%}",
    'CV MAE':    '—',
    'Train MAE': '—',
    'ROC-AUC':   f"{nn_cls_train_auc:.3f}",
    'Val Acc':   f"{cls_val_acc:.1%}",
    'Val AUC':   f"{cls_val_auc:.3f}",
})

# ── Render table ──────────────────────────────────────────────────────────────
summary_df = pd.DataFrame(summary_rows).set_index('Model')

print("\n" + "="*75)
print("  📋 MODEL PERFORMANCE SUMMARY")
print("="*75)
display(summary_df)

# ── Overfitting flag: if Train Acc >> Val Acc, flag the model ─────────────────
print("\n🔍 Overfitting Check:")
for name, train_acc, val_acc in [
    ("Logistic Regression", log_train_acc,    logistic_cv_acc.mean()),
    ("NN Classifier",       nn_cls_train_acc, cls_val_acc),
]:
    gap = train_acc - val_acc
    flag = "⚠️  likely overfitting" if gap > 0.08 else "✅ healthy"
    print(f"   {name:<22}  Train: {train_acc:.1%}  |  Val: {val_acc:.1%}  |  Gap: {gap:+.1%}  {flag}")


  📋 MODEL PERFORMANCE SUMMARY


,Type,Train Acc,CV MAE,Train MAE,ROC-AUC,Val Acc,Val AUC
Model,,,,,,,
Ridge Regression,Regression,59.2%,10.09 ± 1.03,10.76 pts,0.615,—,—
SGD Regressor,Regression,58.1%,9.91 ± 1.10,10.36 pts,0.603,—,—
NN Regressor,Regression,56.2%,N/A,10.50 pts,0.603,—,—
Logistic Regression,Classifier,57.7%,—,—,0.611,57.0% ± 3.6%,0.617 ± 0.034
NN Classifier,Classifier,59.3%,—,—,0.639,57.0%,0.583



🔍 Overfitting Check:
   Logistic Regression     Train: 57.7%  |  Val: 57.0%  |  Gap: +0.7%  ✅ healthy
   NN Classifier           Train: 59.3%  |  Val: 57.0%  |  Gap: +2.4%  ✅ healthy


## Cell 11

In [14]:
# =============================================================================
# CELL 11 — Prediction Functions (all 5 models)
# =============================================================================

def score_diff_to_moneyline(score_diff):
    """
    Converts predicted score differential to American moneyline odds.
    Uses logistic function: each point of margin ~ shifts probability.
    """
    win_prob = 1 / (1 + np.exp(-score_diff / 10.0))
    win_prob = np.clip(win_prob, 0.01, 0.99)
    if win_prob >= 0.5:
        ml = -round((win_prob / (1 - win_prob)) * 100)
    else:
        ml = round(((1 - win_prob) / win_prob) * 100)
    return win_prob, f"+{ml}" if ml > 0 else str(ml)


def prob_to_moneyline(win_prob):
    """Converts a raw win probability directly to American moneyline odds."""
    win_prob = np.clip(win_prob, 0.01, 0.99)
    if win_prob >= 0.5:
        ml = -round((win_prob / (1 - win_prob)) * 100)
    else:
        ml = round(((1 - win_prob) / win_prob) * 100)
    return f"+{ml}" if ml > 0 else str(ml)


def compare_to_spread(predicted_diff, vegas_spread):
    """
    Compares model predicted spread to Vegas spread.
    predicted_diff: positive = home team wins by that many points.
    vegas_spread: negative = home team favored.
    """
    if vegas_spread is None:
        return "N/A", "N/A"
    try:
        spread_val = float(str(vegas_spread).replace('–', '-').strip())
    except Exception:
        return str(vegas_spread), "Parse error"

    model_spread = -round(predicted_diff, 1)

    if model_spread < spread_val:
        edge = "✅ Model likes HOME"
    elif model_spread > spread_val:
        edge = "✅ Model likes AWAY"
    else:
        edge = "⚖️ Neutral"

    return f"Vegas: {spread_val:+.1f} | Model: {model_spread:+.1f}", edge


def predict_game(game, today_team_lookup, numeric_cols):
    """
    Runs all 5 models for a single game and returns a unified result dict.

    Regression models (1-3):  predict score differential
    Classification models (4-5): predict win probability directly

    Ensemble logic:
      - Regression ensemble  : weighted avg of models 1-3 → score diff → implied prob
      - Classification ensemble: weighted avg of models 4-5 → win prob
      - Final ensemble        : average of regression implied prob + classification prob
    """
    home_name = game['home_name']
    away_name = game['away_name']

    X_game        = build_matchup_vector(
        home_name, away_name, today_team_lookup, numeric_cols
    ).reshape(1, -1)
    X_game_scaled = scaler.transform(X_game)

    # ── Regression models → score differential ────────────────────────────────
    lr_diff       = float(lr_model.predict(X_game_scaled)[0])
    sgd_diff      = float(sgd_model.predict(X_game_scaled)[0])
    nn_reg_diff   = float(nn_reg_model.predict(X_game_scaled, verbose=0)[0][0])

    reg_ensemble_diff = round(
        0.3 * lr_diff + 0.3 * sgd_diff + 0.40 * nn_reg_diff, 1
    )

    # Convert regression ensemble diff → implied win probability
    reg_implied_prob, _ = score_diff_to_moneyline(reg_ensemble_diff)

    # ── Classification models → win probability ───────────────────────────────
    logistic_prob = float(logistic_model.predict_proba(X_game_scaled)[0][1])
    nn_cls_prob   = float(nn_cls_model.predict(X_game_scaled, verbose=0)[0][0])

    cls_ensemble_prob = np.clip(
        0.4 * logistic_prob + 0.6 * nn_cls_prob, 0.01, 0.99
    )

    # ── Final ensemble: combine regression implied prob + classification prob ──
    final_win_prob = np.clip(
        0.55 * reg_implied_prob + 0.45 * cls_ensemble_prob, 0.01, 0.99
    )

    # ── Moneylines ────────────────────────────────────────────────────────────
    home_ml_final = prob_to_moneyline(final_win_prob)
    away_ml_final = prob_to_moneyline(1 - final_win_prob)

    # ── Spread analysis ───────────────────────────────────────────────────────
    spread_info, spread_edge = compare_to_spread(
        reg_ensemble_diff, game.get('vegas_spread')
    )

    # ── Confidence ────────────────────────────────────────────────────────────
    # Based on how much all 5 models agree with each other
    all_probs = [
        score_diff_to_moneyline(lr_diff)[0],
        score_diff_to_moneyline(sgd_diff)[0],
        score_diff_to_moneyline(nn_reg_diff)[0],
        logistic_prob,
        nn_cls_prob,
    ]
    prob_std = float(np.std(all_probs))   # low std = models agree = higher confidence

    confidence = (
        'HIGH' if prob_std < 0.08 and abs(final_win_prob - 0.5) > 0.15 else
        'MED'  if prob_std < 0.15 and abs(final_win_prob - 0.5) > 0.08 else
        'LOW'
    )

    # ── Team feature signals for display ─────────────────────────────────────
    h = today_team_lookup.get(home_name, {})
    a = today_team_lookup.get(away_name, {})

    return {
        # Game info
        'Away Team':              away_name,
        'Home Team':              home_name,

        # Regression model outputs (score differential)
        'LR Spread':              f"{lr_diff:+.1f}",
        'SGD Spread':             f"{sgd_diff:+.1f}",
        'NN Reg Spread':          f"{nn_reg_diff:+.1f}",
        'Reg Ensemble Spread':    f"{reg_ensemble_diff:+.1f}",

        # Classification model outputs (win probability)
        'Logistic Win Prob':      f"{logistic_prob:.1%}",
        'NN Cls Win Prob':        f"{nn_cls_prob:.1%}",
        'Cls Ensemble Prob':      f"{cls_ensemble_prob:.1%}",

        # Final combined outputs
        'Final Win Prob (Home)':  f"{final_win_prob:.1%}",
        'Home ML':                home_ml_final,
        'Away ML':                away_ml_final,
        'Predicted Winner':       home_name if final_win_prob >= 0.5 else away_name,
        'Model Agreement':        f"±{prob_std:.3f}",
        'Confidence':             confidence,

        # Spread vs Vegas
        'Spread Analysis':        spread_info,
        'Spread Edge':            spread_edge,

        # Key team signals
        'Home Sentiment':         f"{float(h.get('sent_overall',    0)):+.3f}",
        'Away Sentiment':         f"{float(a.get('sent_overall',    0)):+.3f}",
        'Home Momentum':          f"{float(h.get('momentum_net',    0)):+.3f}",
        'Away Momentum':          f"{float(a.get('momentum_net',    0)):+.3f}",
        'Home Injury Flag':       '🏥 YES' if float(h.get('inj_key_player_flag', 0)) == 1 else 'No',
        'Away Injury Flag':       '🏥 YES' if float(a.get('inj_key_player_flag', 0)) == 1 else 'No',
    }


print("✅ Cell 11 ready — predict_game() pulls from all 5 models.")
print("   Regression  (Models 1–3): LR, SGD, NN Regressor → score differential")
print("   Classification (Models 4–5): Logistic, NN Classifier → win probability")
print("   Final output: 50/50 blend of regression-implied prob + classification prob")

✅ Cell 11 ready — predict_game() pulls from all 5 models.
   Regression  (Models 1–3): LR, SGD, NN Regressor → score differential
   Classification (Models 4–5): Logistic, NN Classifier → win probability
   Final output: 50/50 blend of regression-implied prob + classification prob


# Output

## Cell 12

In [15]:
# =============================================================================
# CELL 12 — Run Predictions for Today's Games (all 5 models)
# =============================================================================

from datetime import date
from IPython.display import display, HTML

# ── Build today's team lookup from feature_matrix (output of Cell 7) ─────────
today_team_lookup = {
    row['team_name']: row.to_dict()
    for _, row in fm_clean.iterrows()
}

print(f"\n🏀 NCAAB PREDICTIONS — {date.today().strftime('%B %d, %Y')}")
print(f"   Training games         : {X_train.shape[0]} real games (Feb 16–22)")
print(f"   Today's teams in lookup: {len(today_team_lookup)}")
print(f"   Games to predict       : {len(games_today)}")
print("=" * 70)

results, skipped = [], []

for game in games_today:
    home_name = game.get('home_name', '')
    away_name = game.get('away_name', '')
    if not home_name or not away_name:
        continue
    if home_name not in today_team_lookup or away_name not in today_team_lookup:
        skipped.append(f"{away_name} @ {home_name}")
        continue
    results.append(predict_game(game, today_team_lookup, numeric_cols))

if skipped:
    print(f"\n⚠️  Skipped {len(skipped)} game(s) — not in today's feature matrix:")
    for s in skipped:
        print(f"   • {s}")

if not results:
    print("❌ No predictions generated.")
else:
    results_df = pd.DataFrame(results)

    def simple_table(df, cols, caption=""):
        """
        Displays a plain styled table with no background_gradient —
        avoids ValueError from percentage/string columns entirely.
        """
        subset = df[cols].reset_index(drop=True)
        styled = (
            subset.style
            .set_caption(caption)
            .set_properties(**{'font-size': '12px', 'text-align': 'center',
                               'padding': '5px 10px'})
            .set_table_styles([
                {'selector': 'th',
                 'props': [('background-color', '#2C3E50'), ('color', 'white'),
                           ('font-size', '12px'), ('padding', '6px 10px'),
                           ('text-align', 'center')]},
                {'selector': 'caption',
                 'props': [('font-size', '13px'), ('font-weight', 'bold'),
                           ('text-align', 'left'), ('padding', '6px 0px')]},
            ])
        )
        return styled

    def color_confidence(val):
        if val == 'HIGH': return 'background-color: #D5F5E3; font-weight: bold'
        if val == 'MED':  return 'background-color: #FCF3CF'
        if val == 'LOW':  return 'background-color: #FADBD8'
        return ''

    def color_spread(val):
        try:
            if isinstance(val, str) and (val.startswith('+') or val.startswith('-')):
                return ('color: #27AE60; font-weight: bold' if val.startswith('+')
                        else 'color: #E74C3C; font-weight: bold')
        except Exception:
            pass
        return ''

    def color_injury(val):
        return 'background-color: #FADBD8; font-weight: bold' if val == '🏥 YES' else ''

    # ── Table 1: Regression model outputs ─────────────────────────────────────
    print("\n📐 REGRESSION MODELS — Predicted Score Differential (Home − Away)\n")
    reg_cols = ['Away Team', 'Home Team',
                'LR Spread', 'SGD Spread', 'NN Reg Spread', 'Reg Ensemble Spread']
    display(
        simple_table(results_df, reg_cols,
                     "Positive = Home team wins by that margin  |  Negative = Away team wins")
        .applymap(color_spread,
                  subset=['LR Spread', 'SGD Spread', 'NN Reg Spread', 'Reg Ensemble Spread'])
    )

    # ── Table 2: Classification model outputs ──────────────────────────────────
    print("\n🎯 CLASSIFICATION MODELS — Home Team Win Probability\n")
    cls_cols = ['Away Team', 'Home Team',
                'Logistic Win Prob', 'NN Cls Win Prob', 'Cls Ensemble Prob']
    display(
        simple_table(results_df, cls_cols, "Win probability for the HOME team")
    )

    # ── Table 3: Final ensemble + moneylines ───────────────────────────────────
    print("\n🏆 FINAL ENSEMBLE — Combined Output from All 5 Models\n")
    final_cols = ['Away Team', 'Home Team',
                  'Reg Ensemble Spread', 'Cls Ensemble Prob',
                  'Final Win Prob (Home)', 'Home ML', 'Away ML',
                  'Predicted Winner', 'Model Agreement', 'Confidence']
    display(
        simple_table(results_df, final_cols,
                     "Final predictions blending regression spread + classification probability")
        .applymap(color_confidence, subset=['Confidence'])
    )

    # ── Table 4: Spread vs Vegas ───────────────────────────────────────────────
    '''print("\n📊 SPREAD ANALYSIS — Model vs Vegas\n")
    spread_cols = ['Away Team', 'Home Team',
                   'Reg Ensemble Spread', 'Final Win Prob (Home)',
                   'Spread Analysis', 'Spread Edge']
    display(simple_table(results_df, spread_cols, "Model spread vs Vegas line"))'''

    # ── Table 5: Team signals ──────────────────────────────────────────────────
    print("\n🔍 TEAM SIGNALS — Sentiment, Momentum & Injury Flags\n")
    signal_cols = ['Away Team', 'Home Team',
                   'Away Sentiment', 'Home Sentiment',
                   'Away Momentum',  'Home Momentum',
                   'Away Injury Flag', 'Home Injury Flag']
    display(
        simple_table(results_df, signal_cols, "Key team signals")
        .applymap(color_injury, subset=['Away Injury Flag', 'Home Injury Flag'])
    )

    # ── Table 6: High confidence picks ────────────────────────────────────────
    high_conf_df = results_df[results_df['Confidence'] == 'HIGH'][[
        'Away Team', 'Home Team', 'Predicted Winner',
        'Final Win Prob (Home)', 'Reg Ensemble Spread',
        'Home ML', 'Away ML', 'Model Agreement', 'Spread Edge'
    ]].reset_index(drop=True)

    print(f"\n🔥 HIGH CONFIDENCE PICKS ({len(high_conf_df)} game(s))\n")
    if not high_conf_df.empty:
        display(
            high_conf_df.style
            .set_properties(**{'background-color': '#EBF5FB',
                               'font-weight': 'bold', 'font-size': '12px',
                               'text-align': 'center'})
            .set_table_styles([{
                'selector': 'th',
                'props': [('background-color', '#1A5276'), ('color', 'white'),
                          ('font-size', '12px'), ('padding', '6px 10px')]
            }])
        )
    else:
        print("   No high-confidence picks today.")

    print("\n" + "=" * 70)
    print(f"   {len(results)} predictions  |  "
          f"{len(results_df[results_df['Confidence']=='HIGH'])} HIGH  |  "
          f"{len(results_df[results_df['Confidence']=='MED'])} MED  |  "
          f"{len(results_df[results_df['Confidence']=='LOW'])} LOW")
    print("=" * 70)
    print("⚠️  DISCLAIMER: Educational/research purposes only.")
    print("    Always gamble responsibly.")
    print("=" * 70)


🏀 NCAAB PREDICTIONS — March 02, 2026
   Training games         : 1382 real games (Feb 16–22)
   Today's teams in lookup: 36
   Games to predict       : 18

📐 REGRESSION MODELS — Predicted Score Differential (Home − Away)



,Away Team,Home Team,LR Spread,SGD Spread,NN Reg Spread,Reg Ensemble Spread
0,Norfolk St.,Morgan St.,+9.3,-5.6,-8.7,-2.4
1,Coppin St.,Howard,-81.1,-6.4,+4.8,-24.4
2,Duke,NC State,-0.3,-2.9,+2.0,-0.2
3,IUI,Clev. St.,+0.0,+2.7,-25.0,-9.2
4,NC Central,Md.-E. Shore,-29.8,+1.6,-14.1,-14.1
5,SC State,Delaware St.,+25.6,+5.1,-6.2,+6.7
6,McNeese,Nicholls,-12.9,-2.2,-4.3,-6.3
7,Northwestern St.,UT-Rio Grande Valley,-33.2,-9.5,+4.7,-10.9
8,SF Austin,Incarnate Word,-74.1,-8.7,+3.5,-23.4
9,Montana,N. Colorado,+32.0,+10.3,-2.2,+11.8



🎯 CLASSIFICATION MODELS — Home Team Win Probability



,Away Team,Home Team,Logistic Win Prob,NN Cls Win Prob,Cls Ensemble Prob
0,Norfolk St.,Morgan St.,82.8%,54.3%,65.7%
1,Coppin St.,Howard,0.0%,95.1%,57.0%
2,Duke,NC State,55.9%,47.5%,50.8%
3,IUI,Clev. St.,50.0%,100.0%,80.0%
4,NC Central,Md.-E. Shore,1.2%,66.0%,40.1%
5,SC State,Delaware St.,98.6%,88.6%,92.6%
6,McNeese,Nicholls,2.0%,0.4%,1.0%
7,Northwestern St.,UT-Rio Grande Valley,0.2%,86.1%,51.7%
8,SF Austin,Incarnate Word,0.0%,98.0%,58.8%
9,Montana,N. Colorado,99.1%,71.8%,82.7%



🏆 FINAL ENSEMBLE — Combined Output from All 5 Models



,Away Team,Home Team,Reg Ensemble Spread,Cls Ensemble Prob,Final Win Prob (Home),Home ML,Away ML,Predicted Winner,Model Agreement,Confidence
0,Norfolk St.,Morgan St.,-2.4,65.7%,53.8%,-116,+116,Morgan St.,±0.203,LOW
1,Coppin St.,Howard,-24.4,57.0%,30.1%,+232,-232,Coppin St.,±0.364,LOW
2,Duke,NC State,-0.2,50.8%,50.1%,-100,+100,NC State,±0.048,LOW
3,IUI,Clev. St.,-9.2,80.0%,51.7%,-107,+107,Clev. St.,±0.293,LOW
4,NC Central,Md.-E. Shore,-14.1,40.1%,28.8%,+247,-247,NC Central,±0.262,LOW
5,SC State,Delaware St.,+6.7,92.6%,78.1%,-356,+356,Delaware St.,±0.237,LOW
6,McNeese,Nicholls,-6.3,1.0%,19.6%,+411,-411,McNeese,±0.183,LOW
7,Northwestern St.,UT-Rio Grande Valley,-10.9,51.7%,37.1%,+169,-169,Northwestern St.,±0.333,LOW
8,SF Austin,Incarnate Word,-23.4,58.8%,31.3%,+220,-220,SF Austin,±0.372,LOW
9,Montana,N. Colorado,+11.8,82.7%,79.3%,-383,+383,N. Colorado,±0.197,LOW



🔍 TEAM SIGNALS — Sentiment, Momentum & Injury Flags



,Away Team,Home Team,Away Sentiment,Home Sentiment,Away Momentum,Home Momentum,Away Injury Flag,Home Injury Flag
0,Norfolk St.,Morgan St.,+0.417,+0.606,+0.000,+0.000,No,No
1,Coppin St.,Howard,+0.966,+0.263,+0.000,+0.050,No,No
2,Duke,NC State,+0.364,+0.231,+0.000,+0.000,No,No
3,IUI,Clev. St.,+0.000,+0.000,+0.000,+0.000,No,No
4,NC Central,Md.-E. Shore,+0.287,+0.000,+0.000,+0.000,No,No
5,SC State,Delaware St.,+0.499,+0.610,+0.000,+0.000,No,No
6,McNeese,Nicholls,-0.008,+0.301,+0.000,-0.050,No,🏥 YES
7,Northwestern St.,UT-Rio Grande Valley,+0.567,+0.220,+0.000,+0.000,No,No
8,SF Austin,Incarnate Word,+0.968,+0.209,+0.000,+0.000,No,No
9,Montana,N. Colorado,+0.113,+0.538,+0.000,+0.000,No,No



🔥 HIGH CONFIDENCE PICKS (0 game(s))

   No high-confidence picks today.

   18 predictions  |  0 HIGH  |  2 MED  |  16 LOW
⚠️  DISCLAIMER: Educational/research purposes only.
    Always gamble responsibly.


## Cell 14

In [16]:
# =============================================================================
# CELL 14 — Daily Betting Optimizer (Top 3 Straight Picks)
# =============================================================================

import itertools
import pandas as pd
import numpy as np
from IPython.display import display, HTML

# ── Constants ─────────────────────────────────────────────────────────────────
SINGLE_BUDGET   = 5.00
MIN_WIN_PROB    = 0.55
MIN_CONF_SINGLE = 'MED'
TOP_N_PICKS     = 3

# ── Helper functions ──────────────────────────────────────────────────────────
def ml_to_decimal(ml_str):
    try:
        ml = int(str(ml_str).replace('+', '').strip())
        return round(1 + ml / 100, 4) if ml > 0 else round(1 + 100 / abs(ml), 4)
    except Exception:
        return None

def decimal_to_payout(decimal_odds, stake):
    if decimal_odds is None:
        return 0.0
    return round((decimal_odds - 1) * stake, 2)

def expected_value(win_prob, net_profit, stake):
    return round((win_prob * net_profit) - ((1 - win_prob) * stake), 4)

def confidence_rank(conf):
    return {'HIGH': 3, 'MED': 2, 'LOW': 1}.get(str(conf), 0)

def confidence_label(rank):
    return {3: '🟢 HIGH', 2: '🟡 MED', 1: '🔴 LOW'}.get(rank, '—')

def parse_win_prob(prob_str):
    try:
        return float(str(prob_str).replace('%', '').strip()) / 100
    except Exception:
        return 0.5


# ── Build eligible candidates ─────────────────────────────────────────────────
candidates = []

for _, row in results_df.iterrows():
    winner     = str(row['Predicted Winner'])
    home_name  = str(row['Home Team'])
    away_name  = str(row['Away Team'])
    confidence = str(row['Confidence'])
    home_prob  = parse_win_prob(row['Final Win Prob (Home)'])

    if winner == home_name:
        bet_side = 'HOME'
        ml_str   = str(row['Home ML'])
        win_prob = home_prob
    else:
        bet_side = 'AWAY'
        ml_str   = str(row['Away ML'])
        win_prob = 1 - home_prob

    if win_prob < MIN_WIN_PROB:
        continue
    if confidence_rank(confidence) < confidence_rank(MIN_CONF_SINGLE):
        continue

    decimal_odds = ml_to_decimal(ml_str)
    if decimal_odds is None or decimal_odds <= 1.0:
        continue

    single_profit = decimal_to_payout(decimal_odds, SINGLE_BUDGET)
    single_ev     = expected_value(win_prob, single_profit, SINGLE_BUDGET)

    candidates.append({
        'matchup':       f"{away_name} @ {home_name}",
        'home_name':     home_name,
        'away_name':     away_name,
        'bet_side':      bet_side,
        'pick':          winner,
        'ml':            ml_str,
        'decimal_odds':  decimal_odds,
        'win_prob':      win_prob,
        'confidence':    confidence,
        'conf_rank':     confidence_rank(confidence),
        'single_profit': single_profit,
        'single_ev':     single_ev,
    })

# ── Sort: confidence first, then EV, then win probability ─────────────────────
candidates_sorted = sorted(
    candidates,
    key=lambda x: (x['conf_rank'], x['single_ev'], x['win_prob']),
    reverse=True
)
top_picks = candidates_sorted[:TOP_N_PICKS]

print(f"  Eligible candidates : {len(candidates)} games")
print(f"  (win prob ≥ {MIN_WIN_PROB:.0%}, confidence ≥ {MIN_CONF_SINGLE})")
print(f"  Showing top {TOP_N_PICKS}\n")


# =============================================================================
# TOP 3 STRAIGHT PICKS
# =============================================================================
print("=" * 65)
print(f"  🎯 TOP {TOP_N_PICKS} STRAIGHT PICKS  —  ${SINGLE_BUDGET:.2f} each")
print("=" * 65)

if not top_picks:
    print("  ❌ No eligible picks today.")
else:
    for rank, pick in enumerate(top_picks, 1):
        print(f"\n  ── Pick #{rank} {'─' * 45}")
        print(f"  Pick       : {pick['pick']}  ({pick['bet_side']})")
        print(f"  Game       : {pick['matchup']}")
        print(f"  ML         : {pick['ml']}  (decimal {pick['decimal_odds']})")
        print(f"  Win Prob   : {pick['win_prob']:.1%}")
        print(f"  Confidence : {confidence_label(pick['conf_rank'])}")
        print(f"  Stake      : ${SINGLE_BUDGET:.2f}")
        print(f"  If Win     : +${pick['single_profit']:.2f}  "
              f"(return ${SINGLE_BUDGET + pick['single_profit']:.2f})")
        print(f"  EV         : ${pick['single_ev']:+.3f}")


# =============================================================================
# SUMMARY TABLE
# =============================================================================
print(f"\n{'=' * 65}")
print(f"  📋 DAILY PICKS SUMMARY  —  Total Budget: ${SINGLE_BUDGET * len(top_picks):.2f}")
print(f"{'=' * 65}")

if top_picks:
    summary_rows = []
    for rank, pick in enumerate(top_picks, 1):
        summary_rows.append({
            'Rank':       f"#{rank}",
            'Pick':       str(pick['pick']),
            'Side':       str(pick['bet_side']),
            'Game':       str(pick['matchup']),
            'ML':         str(pick['ml']),
            'Win Prob':   f"{pick['win_prob']:.1%}",
            'Confidence': confidence_label(pick['conf_rank']),
            'Stake':      f"${SINGLE_BUDGET:.2f}",
            'If Win':     f"+${pick['single_profit']:.2f}",
            'EV':         f"${pick['single_ev']:+.3f}",
        })

    summary_df = pd.DataFrame(summary_rows).set_index('Rank')

    display(
        summary_df.style
        .set_properties(**{'text-align': 'center', 'font-size': '13px',
                           'padding': '6px 12px'})
        .set_table_styles([{
            'selector': 'th',
            'props': [('background-color', '#2C3E50'), ('color', 'white'),
                      ('font-size', '13px'), ('padding', '8px 12px'),
                      ('text-align', 'center')]
        }])
    )

    # ── Scenario breakdown ────────────────────────────────────────────────────
    total_stake  = SINGLE_BUDGET * len(top_picks)
    total_if_all = sum(p['single_profit'] for p in top_picks)
    best_single_win = max(p['single_profit'] for p in top_picks)

    print(f"\n  Total staked            : ${total_stake:.2f}")
    print(f"  If ALL {len(top_picks)} hit             : +${total_if_all:.2f}  "
          f"(return ${total_stake + total_if_all:.2f})")
    print(f"  If best pick (#1) hits  : +${top_picks[0]['single_profit']:.2f}")
    print(f"  If none hit             : -${total_stake:.2f}")

    # ── Combined EV across all 3 picks ───────────────────────────────────────
    combined_ev = sum(p['single_ev'] for p in top_picks)
    print(f"\n  Combined EV (all {len(top_picks)} picks) : ${combined_ev:+.3f}")
    if combined_ev > 0:
        print(f"  ✅ Positive combined EV — mathematically favorable slate")
    else:
        print(f"  ⚠️  Negative combined EV — proceed with caution")

else:
    print("  ❌ No picks recommended today.")

print(f"\n{'=' * 65}")
print(f"  ⚠️  DISCLAIMER: Educational/research purposes only.")
print(f"      Always gamble responsibly. Never bet more than you can afford.")
print(f"{'=' * 65}")

  Eligible candidates : 2 games
  (win prob ≥ 55%, confidence ≥ MED)
  Showing top 3

  🎯 TOP 3 STRAIGHT PICKS  —  $5.00 each

  ── Pick #1 ─────────────────────────────────────────────
  Pick       : Lamar  (AWAY)
  Game       : Lamar @ Houston Chr.
  ML         : -712  (decimal 1.1404)
  Win Prob   : 87.7%
  Confidence : 🟡 MED
  Stake      : $5.00
  If Win     : +$0.70  (return $5.70)
  EV         : $-0.001

  ── Pick #2 ─────────────────────────────────────────────
  Pick       : Arizona  (HOME)
  Game       : Iowa St. @ Arizona
  ML         : -160  (decimal 1.625)
  Win Prob   : 61.5%
  Confidence : 🟡 MED
  Stake      : $5.00
  If Win     : +$3.12  (return $8.12)
  EV         : $-0.006

  📋 DAILY PICKS SUMMARY  —  Total Budget: $10.00


,Pick,Side,Game,ML,Win Prob,Confidence,Stake,If Win,EV
Rank,,,,,,,,,
#1,Lamar,AWAY,Lamar @ Houston Chr.,-712,87.7%,🟡 MED,$5.00,+$0.70,$-0.001
#2,Arizona,HOME,Iowa St. @ Arizona,-160,61.5%,🟡 MED,$5.00,+$3.12,$-0.006



  Total staked            : $10.00
  If ALL 2 hit             : +$3.82  (return $13.82)
  If best pick (#1) hits  : +$0.70
  If none hit             : -$10.00

  Combined EV (all 2 picks) : $-0.007
  ⚠️  Negative combined EV — proceed with caution

  ⚠️  DISCLAIMER: Educational/research purposes only.
      Always gamble responsibly. Never bet more than you can afford.


## Cell 14B

In [17]:
# =============================================================================
# CELL 14B — Feature Explainability — Top 5 drivers for each top pick
# =============================================================================
print(f"\n{'=' * 65}")
print(f"  🔍 WHY THESE PICKS?  —  Top Feature Drivers")
print(f"{'=' * 65}")

def get_top_features_for_matchup(home_name, away_name, top_n=5):
    """
    For a given matchup, builds the feature vector and pulls the top N
    features driving the prediction across all 3 interpretable models
    (Ridge, Gradient Boosting, Logistic). Neural nets are excluded since
    their weights aren't directly interpretable.
    Returns a deduplicated ranked list of (feature_name, avg_importance, direction).
    """
    if home_name not in train_team_lookup or away_name not in train_team_lookup:
        return []

    home_vec    = get_feature_vector(home_name, train_team_lookup, numeric_cols)
    away_vec    = get_feature_vector(away_name, train_team_lookup, numeric_cols)
    diff_vec    = home_vec - away_vec
    is_home     = np.array([0.0])   # zeroed out consistent with training
    feat_vec    = np.concatenate([home_vec, away_vec, diff_vec, is_home])
    feat_scaled = scaler.transform(feat_vec.reshape(1, -1))[0]

    feature_scores = {}

    # ── Ridge: coefficient × feature value = contribution ────────────────────
    ridge_contribs = lr_model.coef_ * feat_scaled
    for i, contrib in enumerate(ridge_contribs):
        fname = feature_names[i]
        if fname == 'is_home':
            continue
        feature_scores.setdefault(fname, []).append(float(contrib))

    # ── Gradient Boosting: feature importance × |feature value| ──────────────
    gb_contribs = sgd_model.feature_importances_ * np.abs(feat_scaled)
    for i, contrib in enumerate(gb_contribs):
        fname = feature_names[i]
        if fname == 'is_home':
            continue
        feature_scores.setdefault(fname, []).append(float(contrib))

    # ── Logistic: coefficient × feature value = log-odds contribution ─────────
    log_contribs = logistic_model.coef_[0] * feat_scaled
    for i, contrib in enumerate(log_contribs):
        fname = feature_names[i]
        if fname == 'is_home':
            continue
        feature_scores.setdefault(fname, []).append(float(contrib))

    # ── Average contribution across models, rank by absolute magnitude ────────
    averaged = []
    for fname, scores in feature_scores.items():
        avg = float(np.mean(scores))
        averaged.append((fname, avg))

    averaged.sort(key=lambda x: abs(x[1]), reverse=True)
    return averaged[:top_n]


def format_feature_name(fname):
    """Converts raw feature name like 'diff_win_pct' into readable label."""
    fname = fname.replace('diff_', 'Δ ').replace('home_', 'Home ').replace('away_', 'Away ')
    fname = fname.replace('_', ' ').title()
    return fname


def explain_pick(rank, pick_name, matchup_str, home_name, away_name, bet_side, win_prob, confidence):
    """Prints the top 5 feature drivers for a single pick."""
    top_feats = get_top_features_for_matchup(home_name, away_name, top_n=5)

    print(f"\n  ── Pick #{rank}: {pick_name}  ({bet_side})  {confidence_label(confidence_rank(confidence))}")
    print(f"     {matchup_str}  |  Win Prob: {win_prob:.1%}")
    print(f"  {'─' * 55}")

    if not top_feats:
        print(f"  ⚠️  Could not compute feature drivers — team not found in training lookup")
        return

    for feat_rank, (fname, avg_contrib) in enumerate(top_feats, 1):
        label     = format_feature_name(fname)
        direction = "favors this pick ✅" if avg_contrib > 0 else "works against ⚠️"
        bar       = "█" * min(int(abs(avg_contrib) * 20), 20)
        print(f"  {feat_rank}. {label:<35} {direction}")
        print(f"     Avg model contribution: {avg_contrib:+.4f}  |{bar}|")


# ── Explain all top picks ─────────────────────────────────────────────────────
if not top_picks:
    print("  ❌ No picks to explain today.")
else:
    for rank, pick in enumerate(top_picks, 1):
        explain_pick(
            rank        = rank,
            pick_name   = str(pick['pick']),
            matchup_str = str(pick['matchup']),
            home_name   = str(pick['home_name']),
            away_name   = str(pick['away_name']),
            bet_side    = str(pick['bet_side']),
            win_prob    = float(pick['win_prob']),
            confidence  = str(pick['confidence'])
        )

print(f"\n{'=' * 65}")
print(f"  ℹ️  Drivers shown are averaged contributions across Ridge,")
print(f"      Gradient Boosting, and Logistic models. Positive values")
print(f"      favor the predicted winner; negative values are risk flags.")
print(f"{'=' * 65}")


  🔍 WHY THESE PICKS?  —  Top Feature Drivers

  ── Pick #1: Lamar  (AWAY)  🟡 MED
     Lamar @ Houston Chr.  |  Win Prob: 87.7%
  ───────────────────────────────────────────────────────
  1. Δ Sent Headlines                    favors this pick ✅
     Avg model contribution: +1.4569  |████████████████████|
  2. Δ Momentum Win Loss Ratio           works against ⚠️
     Avg model contribution: -1.1736  |████████████████████|
  3. Δ Momentum Loss Mention Rate        favors this pick ✅
     Avg model contribution: +0.8168  |████████████████|
  4. Away Sent Headlines                 favors this pick ✅
     Avg model contribution: +0.7955  |███████████████|
  5. Away Momentum Win Loss Ratio        works against ⚠️
     Avg model contribution: -0.7609  |███████████████|

  ── Pick #2: Arizona  (HOME)  🟡 MED
     Iowa St. @ Arizona  |  Win Prob: 61.5%
  ───────────────────────────────────────────────────────
  1. Δ Ctx Home Away Context             favors this pick ✅
     Avg model contribution

## Cell 14C - Articles

In [18]:
# =============================================================================
# CELL 14C — Source Articles — Google News links for top picks
# =============================================================================
print(f"\n{'=' * 65}")
print(f"  📰 SOURCE ARTICLES  —  News Behind Top Picks")
print(f"{'=' * 65}")

def get_articles_for_team(team_name):
    """
    Pulls stored article URLs for a team from the news cache built in Cell 7.
    Falls back to constructing a Google News search URL if cache miss.
    """
    articles = []

    # ── Try to pull from the news cache built during Cell 6/7 ─────────────────
    if 'team_news_cache' in dir() or 'team_news_cache' in globals():
        team_data = team_news_cache.get(team_name, {})

        # Google News RSS articles
        for article in team_data.get('gn_articles', []):
            url    = article.get('url') or article.get('source_url') or article.get('link', '')
            title  = article.get('title', 'No title')
            source = article.get('source', 'Google News')
            if url and url.startswith('http'):
                articles.append({'title': title, 'url': url, 'source': source})

        # ESPN articles
        for article in team_data.get('espn_articles', []):
            url   = article.get('url') or article.get('source_url') or article.get('link', '')
            title = article.get('title', 'No title')
            if url and url.startswith('http'):
                articles.append({'title': title, 'url': url, 'source': 'ESPN'})

        # Injury articles
        for article in team_data.get('injury_articles', []):
            url   = article.get('url') or article.get('source_url') or article.get('link', '')
            title = article.get('title', 'No title')
            if url and url.startswith('http'):
                articles.append({'title': title, 'url': url, 'source': 'ESPN (Injury)'})

    # ── Fallback: construct Google News search URL if no cache found ───────────
    if not articles:
        encoded_team = requests.utils.quote(f"{team_name} mens college basketball NCAAB")
        fallback_url = f"https://news.google.com/search?q={encoded_team}&hl=en-US"
        articles.append({
            'title':  f"Google News search: {team_name}",
            'url':    fallback_url,
            'source': 'Google News (search)'
        })

    # ── Deduplicate by URL ────────────────────────────────────────────────────
    seen_urls = set()
    deduped   = []
    for a in articles:
        if a['url'] not in seen_urls:
            seen_urls.add(a['url'])
            deduped.append(a)

    return deduped[:5]   # cap at 5 articles per team


def print_articles_for_pick(rank, pick_name, home_name, away_name, matchup_str, win_prob, confidence):
    """Prints article links for both teams in a matchup."""
    print(f"\n  ── Pick #{rank}: {pick_name}  |  Win Prob: {win_prob:.1%}  "
          f"|  {confidence_label(confidence_rank(confidence))}")
    print(f"     {matchup_str}")
    print(f"  {'─' * 55}")

    for team_name, label in [(home_name, 'Home'), (away_name, 'Away')]:
        articles = get_articles_for_team(team_name)
        print(f"\n  {label}: {team_name}")
        if not articles:
            print(f"    • No articles found")
        else:
            for i, article in enumerate(articles, 1):
                title  = str(article['title'])[:80]
                source = article['source']
                url    = article['url']
                print(f"    {i}. [{source}] {title}")
                print(f"       🔗 {url}")


# ── Print articles for all top picks ─────────────────────────────────────────
if not top_picks:
    print("  ❌ No picks to show articles for today.")
else:
    for rank, pick in enumerate(top_picks, 1):
        print_articles_for_pick(
            rank        = rank,
            pick_name   = str(pick['pick']),
            home_name   = str(pick['home_name']),
            away_name   = str(pick['away_name']),
            matchup_str = str(pick['matchup']),
            win_prob    = float(pick['win_prob']),
            confidence  = str(pick['confidence'])
        )

print(f"\n{'=' * 65}")
print(f"  ℹ️  Articles sourced from Google News RSS + ESPN fetched in Cell 7.")
print(f"      Links open the original publisher page, not a Google cache.")
print(f"      Fallback links open a Google News men's basketball search.")
print(f"{'=' * 65}")


  📰 SOURCE ARTICLES  —  News Behind Top Picks

  ── Pick #1: Lamar  |  Win Prob: 87.7%  |  🟡 MED
     Lamar @ Houston Chr.
  ───────────────────────────────────────────────────────

  Home: Houston Chr.
    1. [Google News (search)] Google News search: Houston Chr.
       🔗 https://news.google.com/search?q=Houston%20Chr.%20mens%20college%20basketball%20NCAAB&hl=en-US

  Away: Lamar
    1. [Google News (search)] Google News search: Lamar
       🔗 https://news.google.com/search?q=Lamar%20mens%20college%20basketball%20NCAAB&hl=en-US

  ── Pick #2: Arizona  |  Win Prob: 61.5%  |  🟡 MED
     Iowa St. @ Arizona
  ───────────────────────────────────────────────────────

  Home: Arizona
    1. [Google News (search)] Google News search: Arizona
       🔗 https://news.google.com/search?q=Arizona%20mens%20college%20basketball%20NCAAB&hl=en-US

  Away: Iowa St.
    1. [Google News (search)] Google News search: Iowa St.
       🔗 https://news.google.com/search?q=Iowa%20St.%20mens%20college%20basketb